### BERT pretrained model 제작 [프로젝트]
1. 프로젝트 과정
    1. Step 1: 데이터 전처리 및 토큰화
        1. Corpus 정제: 한국어 위키백과(kowiki.txt)를 대상으로 정규표현식을 활용해 한자, URL, 이메일, 특수 기호 등을 제거하여 고품질의 텍스트 데이터셋을 구축했습니다.

        2. Tokenizer 구성: SentencePiece 기반 unigram 토크나이저를 사용했습니다.
            - Vocab Size: 8,000
            - Special Tokens: [PAD], [UNK], [BOS], [EOS], [SEP], [CLS], [MASK]

        3. Pretrain 데이터셋 생성 (MLM & NSP):
            - MLM: 전체 토큰의 15%를 마스킹하며, 그중 80%는 [MASK], 10%는 Random, 10%는 Original 유지

            - NSP: 50:50 비율로 실제 연속된 문장(IsNext)과 무작위 문장(NotNext)을 구성했습니다.

            - 효율화: np.memmap을 적용하여 대용량 데이터를 메모리 점유 없이 처리하는 디스크 기반 매핑을 구현했습니다.

    2. Step 2: 학습 구성 (Optimizer & Scheduler)
        1. 초소형 모델 아키텍처 (Mini-BERT):
            - 파라미터: 약 1M
            - 하이퍼파라미터: $n_{layer}=3$, $d_{model}=90$, $n_{head}=5$, $d_{ff}=360$ 
            - 최신 기법: 학습 안정성을 위해 RMSNorm과 SwiGLU 활성화 함수를 적용했습니다.

        2. 최적화 전략:
            - Optimizer: AdamW (Weight Decay 0.01 적용, 단 Bias/LayerNorm은 제외).
            - Scheduler: Warmup이 포함된 Cosine Annealing 스케줄러를 직접 구현했습니다.
            - Efficiency: **Mixed Precision(torch.amp)**을 통해 연산 속도를 가속화했습니다.

    3. Step 3: 주요 평가지표(Metrics) 정의
        - 손실 함수 (Loss): NSP는 CrossEntropy를, MLM은 Label Smoothing이 적용된 CrossEntropy(lm_loss)를 사용했으며, 두 Loss를 합산하기 위해 MultiTaskLoss 모듈을 정의했습니다.

        - 평가지표: Total Loss, NSP Loss, MLM Loss의 감소 추이와 함께, NSP Accuracy, MLM Accuracy, MLM Perplexity(PPL), NSP F1 Score, NSP ROC-AUC 등 다양한 지표를 산출하여 학습 과정을 다각도로 모니터링했습니다.
        
            - NSP Accuracy: 두 문장의 문맥적 연결을 맞춘 비율
                - 높을수록 모델이 문장의 흐름과 문맥을 잘 이해함을 의미합니다.

            - MLM Accuracy: 빈칸에 들어갈 올바른 단어를 맞춘 비율
                - 높을수록 언어의 통계적 구조와 단어 간 관계를 잘 파악한 것입니다.

            - MLM Perplexity(PPL): 모델이 다음 단어를 예측할 때의 헷갈림 정도
                - 낮을수록 모델이 정답에 대해 확신을 가지고 정확하게 예측하고 있다는 증거입니다.

            - NSP F1 Score / NSP ROC-AUC: 정밀도와 재현율을 고려한 이진 분류 지표
                - 높을수록 단순 정확도를 넘어 클래스 불균형에도 견고한 판단 능력을 갖췄음을 뜻합니다.

    4. Step 4: 정성적 평가 (Inference Test)
        - 학습된 모델을 활용하여 5가지 카테고리(일반 지식, 일상, 논리, 감정, 이상치)에 대한 추론을 수행하고, klue/bert-base 모델과 비교하여 성능 차이를 분석했습니다.

2. 프로젝트 문제점 및 개선 방안
    - Q1. MLM Loss Scale 최적값 탐색
        - 현상: 두 태스크 간 영향력 균형을 위한 스케일링 필요성 대두.
    
        - 해결: 실험 결과 Scale 1.0이 가장 안정적이었으며, 동적 스케일링은 오히려 학습 진동을 유발함을 확인했습니다.

    - Q2. 하드웨어 제약(VRAM) 극복
        - 현상: 권장 배치 사이즈(256 이상) 확보 불가능(물리적 한계 64).

        - 해결: Gradient Accumulation(Batch 64 × Accumulate 4)을 통해 논리적 배치 256의 효과를 구현했습니다.

    - Q3. 효율적인 학습 데이터 및 스텝 수
        - 현상: 데이터량과 학습 시간 사이의 가성비 지점 파악의 어려움.

        - 해결: 1,800 Step 대비 5,000 Step 학습 시 성능이 확연히 개선됨을 확인하여 학습량 증설의 유효성을 검증했습니다. 
            * 모델의 수렴 지점은 5,000 Step 이상일 가능성이 높습니다.

3. 주요 지표 및 성능 분석
    1. 정량적 지표 분석 (Quantitative Metrics)
        - v2 모델 (Scale 1.0): NSP Accuracy 0.68, ROC-AUC 0.80으로 문맥 파악 능력 최대화.
        - v3 모델 (Scale 10.0): MLM Accuracy 0.23, PPL 240으로 빈칸 예측에서 가장 정교한 성능 기록.
        - 학습 안정성: 동적 스케일링(v5)은 후반부에 NSP Loss가 0.60 이상으로 튀며 불안정한 양상을 보였습니다.
    
    2. 정성적 평가 결과 (Qualitative Evaluation)
        - 추론 성능: v2 모델이 미학습 문장에 대해 64%의 정답률을 보이며 실전 문맥 파악력이 우수했습니다.
        - 생성 잠재력: v3 모델은 비록 [UNK] 비중이 높았으나, '__또는', '__수' 등 문법적으로 유의미한 예측을 시도하는 모습을 보였습니다.
        - 결론: MLM 스케일 조정이 역전파되는 그래디언트 강도를 변화시켜 전체 모델의 태스크 편향성에 직결됨을 확인했습니다.

4. 추가 개선 방향
    1. 지식 증류 (Knowledge Distillation): klue/bert-base를 Teacher로 설정하여 1M 크기의 모델에 풍부한 지식을 전이.
    2. 구조적 Sweet Spot 탐색: 리소스가 허용하는 범위 내에서 파라미터를 5M~15M까지 확장하며 성능 향상 폭 측정.

### 회고  
미니 모델을 직접 만들고 추론해 보는 과정 자체는 수월했지만, 이 모델의 성능을 어떤 기준으로 평가해야 할지 척도를 정하는 데 어려움이 있었으며, 앞으로 이 작은 모델을 구체적으로 어떤 태스크에 유용하게 활용할 수 있을지에 대해 깊이 고민하게 된 프로젝트였습니다.

### [최종 요약 결론]
본 프로젝트는 제한된 자원(1M 파라미터, 배치 사이즈 64) 환경에서 Gradient Accumulation과 Loss Scaling 실험을 통해 한국어 BERT 모델을 성공적으로 구현했습니다. 실험 결과, NSP 중심의 v2 모델과 MLM 중심의 v3 모델이 각각의 강점을 보였으며, 향후 지식 증류(Distillation) 기법을 도입한다면 소형 모델의 한계를 극복하고 실용적인 성능을 확보할 수 있을 것으로 판단됩니다.

In [1]:
# !mkdir -p ./data
# !mkdir -p ./models
# !wget https://aiffelstaticprd.blob.core.windows.net/media/documents/kowiki.txt.zip
# !mv kowiki.txt.zip ./data
# !cd ./data && unzip kowiki.txt.zip
# !unzip ./data/kowiki.txt.zip

In [2]:
# #다음 라이브러리를 설치해주세요
# !pip install sentencepiece
# !pip install tqdm
# !conda install -y -c conda-forge ipywidgets
# !jupyter nbextension enable --py widgetsnbextension
# !pip install torchsummary torchinfo

### 라이브러리 임포트 및 기본 경로 설정

In [1]:
# imports
from __future__ import absolute_import, division, print_function, unicode_literals

import os
import re
import math
import time
import numpy as np
import random
import collections
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import f1_score, roc_auc_score
from torchinfo import summary

import matplotlib.pyplot as plt
import sentencepiece as spm
from tqdm import tqdm

# 재현성을 위한 시드 고정
random_seed = 1234
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)

print(f"Torch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Torch Version: 2.10.0+cu128
Device: cuda


In [2]:
# 외부 변수로 파일 경로 선언
DATA_DIR = './data'
MODELS_DIR = './models'
CORPUS_FILE = os.path.join(DATA_DIR, 'kowiki.txt')
PREFIX = os.path.join(DATA_DIR, 'ko_8000_uni')
PRETRAIN_JSON_PATH = './bert_pre_train.json'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

### Step 1 하이퍼파라미터 설정

In [3]:
class Config:
    def __init__(self, config_dict):
        for key, value in config_dict.items():
            setattr(self, key, value)

def get_base_config(n_layer=3):
    """
    레이어 수(n_layer)를 유연하게 조절하여 모델 크기를 변경할 수 있는 설정 생성 함수
    """
    return Config({
        "epochs": 10,
        "batch_size": 64,
        "n_vocab": 8000,
        "d_model": 90,
        "n_head": 5,
        "dropout": 0.1,
        "d_ff": 360,
        "layernorm_epsilon": 1e-5,
        "n_layer": n_layer, # 다중 모델 비교를 위한 가변 파라미터
        "use_shared_layer": False, 
        "n_seq": 128,
        "i_pad": 0
    })

config = get_base_config()

### Step 2~4: 데이터 전처리 및 Dataloader

In [4]:
def clean_wiki_corpus(input_path, output_path):
    # 정규표현식 패턴 미리 컴파일 (속도 향상)
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    email_pattern = re.compile(r'[a-zA-Z0-9+-_.]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+')
    
    # 위키백과 특유의 빈 괄호 찌꺼기 제거용 패턴 (예: () 혹은 [] )
    empty_bracket_pattern1 = re.compile(r'\(\s*\)')
    empty_bracket_pattern2 = re.compile(r'\[\s*\]')
    
    # 연속된 공백이나 탭을 하나의 공백으로 치환하는 패턴
    space_pattern = re.compile(r'\s+')

    line_cnt = sum(1 for _ in open(input_path, 'r', encoding='utf-8'))
    
    with open(input_path, 'r', encoding='utf-8') as f_in, \
         open(output_path, 'w', encoding='utf-8') as f_out:
        
        for line in tqdm(f_in, total=line_cnt, desc="Cleaning Corpus"):
            text = line.strip()
            
            # 빈 줄은 단락 구분을 위해 그대로 유지
            if not text:
                f_out.write('\n')
                continue
            
            # 1. URL 및 이메일 제거
            text = url_pattern.sub('', text)
            text = email_pattern.sub('', text)
            
            # 2. 내용이 비어버린 괄호 제거
            text = empty_bracket_pattern1.sub('', text)
            text = empty_bracket_pattern2.sub('', text)
            
            # 3. 특수문자 전처리 (필요시 주석 해제하여 사용)
            # 한글, 영문, 숫자, 기본 문장부호만 남기고 싶다면 아래 코드를 활성화하세요.
            text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9.,?!%\'\"()\[\]\- ]', ' ', text)
            
            # 4. 여러 개의 공백을 하나의 공백으로 압축
            text = space_pattern.sub(' ', text).strip()
            
            if text: # 완전히 지워지지 않은 텍스트만 기록
                f_out.write(text + '\n')
                
    print(f"전처리 완료! 깨끗한 코퍼스가 저장되었습니다: {output_path}")

# 파일 경로 설정 및 함수 실행
CLEAN_CORPUS_FILE = os.path.join(DATA_DIR, 'kowiki_clean.txt')

# 파일이 없을 때만 실행하도록 방어 코드 추가
if not os.path.exists(CLEAN_CORPUS_FILE):
    clean_wiki_corpus(CORPUS_FILE, CLEAN_CORPUS_FILE)
else:
    print("이미 전처리된 파일이 존재합니다.")

이미 전처리된 파일이 존재합니다.


In [5]:
# # 센텐스피스를 직접 만들어보실 분들은 주석을 풀어 진행해보세요! 
# #(LMS 환경에서는 시간이 오래 걸려요)
# spm.SentencePieceTrainer.train(f"--input={CLEAN_CORPUS_FILE} --model_prefix={PREFIX} --vocab_size={config.n_vocab + 7} --model_type=bpe --max_sentence_length=999999 --pad_id=0 --pad_piece=[PAD] --unk_id=1 --unk_piece=[UNK] --bos_id=2 --bos_piece=[BOS] --eos_id=3 --eos_piece=[EOS] --user_defined_symbols=[SEP],[CLS],[MASK]")

In [6]:
vocab = spm.SentencePieceProcessor()
vocab.load(f"{PREFIX}.model")
config.n_vocab = len(vocab)
config.i_pad = vocab.piece_to_id('[PAD]')

In [7]:
# 0번부터 6번까지 어떤 토큰인지 직접 찍어보기
for i in range(7):
    print(f"Index {i}: {vocab.id_to_piece(i)}")
len(vocab)

Index 0: [PAD]
Index 1: [UNK]
Index 2: [BOS]
Index 3: [EOS]
Index 4: [SEP]
Index 5: [CLS]
Index 6: [MASK]


8007

In [8]:
def create_pretrain_mask(tokens, mask_cnt, vocab_list):
    # 단어 단위로 mask 하기 위해서 index 분할 (띄어쓰기 기준)
    cand_idx = []
    for (i, token) in enumerate(tokens):
        if token == "[CLS]" or token == "[SEP]":
            continue
        if 0 < len(cand_idx) and not token.startswith(u"\u2581"):
            cand_idx[-1].append(i)
        else:
            cand_idx.append([i])

    random.shuffle(cand_idx)
    mask_lms = []
    for index_set in cand_idx:
        if len(mask_lms) >= mask_cnt:
            break
        if len(mask_lms) + len(index_set) > mask_cnt:
            continue
        dice = random.random()
        for index in index_set:
            if dice < 0.8:
                masked_token = "[MASK]"
            elif dice < 0.9:
                masked_token = tokens[index]
            else:
                masked_token = random.choice(vocab_list)
            mask_lms.append({"index": index, "label": tokens[index]})
            tokens[index] = masked_token

    mask_lms = sorted(mask_lms, key=lambda x: x["index"])
    mask_idx = [p["index"] for p in mask_lms]
    mask_label = [p["label"] for p in mask_lms]

    return tokens, mask_idx, mask_label

def trim_tokens(tokens_a, tokens_b, max_seq):
    while True:
        total_length = len(tokens_a) + len(tokens_b)
        if total_length <= max_seq:
            break
        if len(tokens_a) > len(tokens_b):
            del tokens_a[0]
        else:
            tokens_b.pop()

def create_pretrain_instances(vocab, doc, n_seq, mask_prob, vocab_list):
    max_seq = n_seq - 3 # [CLS], [SEP], [SEP] 제외
    instances = []
    current_chunk = []
    current_length = 0
    
    for i in range(len(doc)):
        current_chunk.append(doc[i])
        current_length += len(doc[i])
        if 1 < len(current_chunk) and (i == len(doc) - 1 or current_length >= max_seq):
            a_end = random.randrange(1, len(current_chunk)) if 1 < len(current_chunk) else 1
            
            tokens_a = [tok for chunk in current_chunk[:a_end] for tok in chunk]
            tokens_b = [tok for chunk in current_chunk[a_end:] for tok in chunk]

            if random.random() < 0.5:
                is_next = 0
                tokens_a, tokens_b = tokens_b, tokens_a
            else:
                is_next = 1
                
            trim_tokens(tokens_a, tokens_b, max_seq)
            tokens = ["[CLS]"] + tokens_a + ["[SEP]"] + tokens_b + ["[SEP]"]
            segment = [0] * (len(tokens_a) + 2) + [1] * (len(tokens_b) + 1)
            tokens, mask_idx, mask_label = create_pretrain_mask(tokens, int((len(tokens) - 3) * mask_prob), vocab_list)

            instances.append({
                "tokens": tokens,
                "segment": segment,
                "is_next": is_next,
                "mask_idx": mask_idx,
                "mask_label": mask_label
            })
            current_chunk = []
            current_length = 0
    return instances

In [9]:
def make_pretrain_data(vocab, in_file, out_file, n_seq, mask_prob=0.15):
    if os.path.exists(out_file):
        print(f"'{out_file}' 파일이 이미 존재합니다.")
        return
    
    """ pretrain 데이터 생성 """
    def save_pretrain_instances(out_f, doc):
        instances = create_pretrain_instances(vocab, doc, n_seq, mask_prob, vocab_list)
        for instance in instances:
            out_f.write(json.dumps(instance, ensure_ascii=False))
            out_f.write("\n")

    # 특수문자 7개를 제외한 vocab_list 생성
    vocab_list = []
    for id in range(7, len(vocab)):
        if not vocab.is_unknown(id):        # 생성되는 단어 목록이 unknown인 경우는 제거합니다.
            vocab_list.append(vocab.id_to_piece(id))

    # line count 확인
    line_cnt = 0
    with open(in_file, "r") as in_f:
        for line in in_f:
            line_cnt += 1

    with open(in_file, "r") as in_f:
        with open(out_file, "w") as out_f:
            doc = []
            for line in tqdm(in_f, total=line_cnt):
                line = line.strip()
                if line == "":  # line이 빈줄 일 경우 (새로운 단락)
                    if 0 < len(doc):
                        save_pretrain_instances(out_f, doc)
                        doc = []
                else:  # line이 빈줄이 아닐 경우 tokenize 해서 doc에 저장
                    pieces = vocab.encode_as_pieces(line)
                    if 0 < len(pieces):
                        doc.append(pieces)
            if 0 < len(doc):  # 마지막에 처리되지 않은 doc가 있는 경우
                save_pretrain_instances(out_f, doc)
                doc = []
                
make_pretrain_data(vocab, CORPUS_FILE, PRETRAIN_JSON_PATH, 128)

'./bert_pre_train.json' 파일이 이미 존재합니다.


In [10]:
def load_pre_train_data(vocab, filename, n_seq, count=None):
    """
    학습에 필요한 데이터를 로드
    :param vocab: vocab
    :param filename: 전처리된 json 파일
    :param n_seq: 시퀀스 길이 (number of sequence)
    :param count: 데이터 수 제한 (None이면 전체)
    :return enc_tokens: encoder inputs
    :return segments: segment inputs
    :return labels_nsp: nsp labels
    :return labels_mlm: mlm labels
    """
    total = 0
    with open(filename, "r") as f:
        for line in f:
            total += 1
            # 데이터 수 제한
            if count is not None and count <= total:
                break

    # np.memmap을 사용하면 메모리를 적은 메모리에서도 대용량 데이터 처리가 가능 함
    enc_tokens = np.memmap(filename='enc_tokens.memmap', mode='w+', dtype=np.int32, shape=(total, n_seq))
    segments = np.memmap(filename='segments.memmap', mode='w+', dtype=np.int32, shape=(total, n_seq))
    labels_nsp = np.memmap(filename='labels_nsp.memmap', mode='w+', dtype=np.int32, shape=(total,))
    labels_mlm = np.memmap(filename='labels_mlm.memmap', mode='w+', dtype=np.int32, shape=(total, n_seq))

    with open(filename, "r") as f:
        for i, line in enumerate(tqdm(f, total=total)):
            if total <= i:
                print("data load early stop", total, i)
                break
            data = json.loads(line)
            # encoder token
            enc_token = [vocab.piece_to_id(p) for p in data["tokens"]]
            enc_token += [0] * (n_seq - len(enc_token))
            # segment
            segment = data["segment"]
            segment += [0] * (n_seq - len(segment))
            # nsp label
            label_nsp = data["is_next"]
            # mlm label
            mask_idx = np.array(data["mask_idx"], dtype=np.int32)
            mask_label = np.array([vocab.piece_to_id(p) for p in data["mask_label"]], dtype=np.int32)
            label_mlm = np.full(n_seq, dtype=np.int32, fill_value=0)
            label_mlm[mask_idx] = mask_label

            assert len(enc_token) == len(segment) == len(label_mlm) == n_seq

            enc_tokens[i] = enc_token
            segments[i] = segment
            labels_nsp[i] = label_nsp
            labels_mlm[i] = label_mlm

    return (enc_tokens, segments), (labels_nsp, labels_mlm)

def load_datasets(pretrain_json_path, config, count=128000):
    """데이터 로드 및 분할을 수행하여 DataLoader를 반환합니다."""
    # 데이터 로드 (미리 정의된 load_pre_train_data 활용)
    pre_train_inputs, pre_train_labels = load_pre_train_data(vocab, pretrain_json_path, config.n_seq, count=count)
    
    pre_train_inputs_t = [torch.tensor(np.array(x)) for x in pre_train_inputs]
    pre_train_labels_t = [torch.tensor(np.array(x)) for x in pre_train_labels]
    full_dataset = TensorDataset(pre_train_inputs_t[0], pre_train_inputs_t[1], pre_train_labels_t[0], pre_train_labels_t[1])
    
    val_size = int(len(full_dataset) * 0.1)
    train_size = len(full_dataset) - val_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, pin_memory=True, num_workers=2, drop_last=True)
    
    return train_loader, val_loader, vocab

In [11]:
#전체 데이터 수 3957761
# batch size: 64, train: val > 9:1
# 355555: 5000step, 128000: 1800step
train_loader, val_loader, vocab = load_datasets(PRETRAIN_JSON_PATH, config, count=355555)

100%|██████████| 355555/355555 [00:20<00:00, 16956.76it/s]


data load early stop 355555 355555


### Step 5: 모델 아키텍처

In [12]:
def get_pad_mask(tokens, i_pad=0):
    """
    pad mask 계산하는 함수
    :param tokens: tokens (bs, n_seq)
    :param i_pad: id of pad
    :return mask: pad mask (pad: 1, other: 0)
    """
    mask = (tokens == i_pad).float()
    mask = mask.unsqueeze(1)
    return mask

class SharedEmbedding(nn.Module):
    """ Weighed Shared Embedding Class """
    def __init__(self, config, name="weight_shared_embedding"):
        super().__init__()
        self.n_vocab = config.n_vocab
        self.d_model = config.d_model
        self.shared_weights = nn.Parameter(torch.empty(self.n_vocab, self.d_model))
        nn.init.trunc_normal_(self.shared_weights, std=0.02)

    def forward(self, inputs, mode="embedding"):
        if mode == "embedding":
            return self._embedding(inputs)
        elif mode == "linear":
            return self._linear(inputs)

    def _embedding(self, inputs):
        inputs = torch.clamp(inputs, max=self.shared_weights.size(0) - 1)
        return self.shared_weights[inputs.long()]

    def _linear(self, inputs):
        n_batch, n_seq, _ = inputs.shape
        inputs = inputs.view(-1, self.d_model)
        outputs = torch.matmul(inputs, self.shared_weights.T)
        outputs = outputs.view(n_batch, n_seq, self.n_vocab)
        return outputs

class PositionEmbedding(nn.Module):
    """ Position Embedding Class """
    def __init__(self, config, name="position_embedding"):
        super().__init__()
        self.embedding = nn.Embedding(config.n_seq, config.d_model)
        nn.init.trunc_normal_(self.embedding.weight, std=0.02)

    def forward(self, inputs):
        position = torch.cumsum(torch.ones_like(inputs), dim=1) - 1
        position = position.long()
        embed = self.embedding(position)
        return embed

class ScaleDotProductAttention(nn.Module):
    def __init__(self, name="scale_dot_product_attention"):
        super().__init__()

    def forward(self, Q, K, V, attn_mask):
        attn_score = torch.matmul(Q, K.transpose(-2, -1))
        scale = torch.sqrt(torch.tensor(K.shape[-1], dtype=torch.float32))
        attn_scale = attn_score / scale
        attn_scale = attn_scale - (attn_mask * 1e9)
        attn_prob = F.softmax(attn_scale, dim=-1)
        attn_out = torch.matmul(attn_prob, V)
        return attn_out

class MultiHeadAttention(nn.Module):
    def __init__(self, config, name="multi_head_attention"):
        super().__init__()
        self.d_model = config.d_model
        self.n_head = config.n_head
        self.d_head = self.d_model // self.n_head

        # [방어 코드 추가] d_model이 n_head로 나누어떨어지지 않으면 에러를 뱉습니다.
        assert self.d_model % self.n_head == 0, \
            f"d_model({self.d_model})은 n_head({self.n_head})로 나누어떨어져야 합니다!"

        self.W_Q = nn.Linear(self.d_model, self.n_head * self.d_head)
        self.W_K = nn.Linear(self.d_model, self.n_head * self.d_head)
        self.W_V = nn.Linear(self.d_model, self.n_head * self.d_head)
        self.W_O = nn.Linear(self.n_head * self.d_head, self.d_model)
        
        self.attention = ScaleDotProductAttention()

    def forward(self, Q, K, V, attn_mask):
        batch_size = Q.shape[0]
        Q_m = self.W_Q(Q).view(batch_size, -1, self.n_head, self.d_head).transpose(1, 2)
        K_m = self.W_K(K).view(batch_size, -1, self.n_head, self.d_head).transpose(1, 2)
        V_m = self.W_V(V).view(batch_size, -1, self.n_head, self.d_head).transpose(1, 2)
        attn_mask_m = attn_mask.unsqueeze(1)
        attn_out = self.attention(Q_m, K_m, V_m, attn_mask_m)
        attn_out_m = attn_out.transpose(1, 2).contiguous() 
        attn_out = attn_out_m.view(batch_size, -1, self.n_head * self.d_head) 
        attn_out = self.W_O(attn_out)
        return attn_out

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        # 평균 계산 없이 제곱의 평균(분산)만으로 정규화하여 연산 효율 극대화
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x

class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        # 파라미터 수를 기존 FFN과 유사하게 맞추기 위해 d_ff를 축소 (약 2/3 수준)
        hidden_dim = int(config.d_ff * (2 / 3)) 
        self.w1 = nn.Linear(config.d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(config.d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, config.d_model, bias=False)

    def forward(self, x):
        # Swish 활성화 함수 (F.silu) 와 게이트 연산의 결합
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class EncoderLayer(nn.Module):
    def __init__(self, config, name="encoder_layer"):
        super().__init__()
        # 기존에 선언한 MultiHeadAttention 클래스 활용
        self.self_attention = MultiHeadAttention(config) 
        self.norm1 = RMSNorm(config.d_model, eps=config.layernorm_epsilon)
        self.ffn = SwiGLU(config)
        self.norm2 = RMSNorm(config.d_model, eps=config.layernorm_epsilon)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, enc_embed, self_mask):
        # Pre-LN 구조 유지: 연산 전 RMSNorm 통과
        norm1_val = self.norm1(enc_embed)
        self_attn_val = self.self_attention(norm1_val, norm1_val, norm1_val, self_mask)
        attn_out = enc_embed + self.dropout(self_attn_val)

        norm2_val = self.norm2(attn_out)
        ffn_val = self.ffn(norm2_val)
        enc_out = attn_out + self.dropout(ffn_val)
        
        return enc_out

class BERT(nn.Module):
    """
    BERT Class
    """
    def __init__(self, config):
        """
        생성자
        :param config: Config 객체
        """
        super(BERT, self).__init__()
        self.i_pad = config.i_pad
        self.embedding = SharedEmbedding(config)
        self.position = PositionEmbedding(config)
        self.segment = nn.Embedding(2, config.d_model)
        self.norm = nn.LayerNorm(config.d_model, eps=config.layernorm_epsilon)

        self.encoder_layers = nn.ModuleList([EncoderLayer(config, name=f"encoder_layer_{i}") for i in range(config.n_layer)])
        self.dropout = nn.Dropout(config.dropout)

        self.final_norm = RMSNorm(config.d_model, eps=config.layernorm_epsilon)

        self.lm_transform = nn.Sequential(
            nn.Linear(config.d_model, config.d_model),
            nn.GELU(), # SwiGLU와 호환성이 좋은 GELU 사용
            RMSNorm(config.d_model, eps=config.layernorm_epsilon)
        )

    def forward(self, enc_tokens, segments):
        """
        layer 실행
        :param enc_tokens: 입력 token들
        :param segments: 입력 segment들
        :return logits_cls: CLS 토큰에 대한 예측
        :return logits_lm: Masked Language Modeling 예측
        """
        enc_self_mask = get_pad_mask(enc_tokens, self.i_pad)
        enc_embed = self.get_embedding(enc_tokens, segments)
        enc_out = self.dropout(enc_embed)

        for encoder_layer in self.encoder_layers:
            enc_out = encoder_layer(enc_out, enc_self_mask)

        # 1. [핵심] Pre-LN의 필수 마무리: 최종 정규화 (이게 없으면 동적 마스킹 감당 불가)
        enc_out = self.final_norm(enc_out) 

        logits_cls = enc_out[:, 0]
        
        # 2. [핵심] LM Head: 문맥을 단어로 번역해주는 층 (이게 있어야 진짜 실력이 늠)
        lm_hidden = self.lm_transform(enc_out) 
        logits_lm = self.embedding(lm_hidden, mode="linear")
        
        return logits_cls, logits_lm

    def get_embedding(self, tokens, segments):
        """
        token embedding, position embedding lookup
        :param tokens: 입력 tokens
        :param segments: 입력 segments
        :return embed: embedding 결과
        """
        embed = self.embedding(tokens) + self.position(tokens) + self.segment(segments)
        embed = self.norm(embed)
        return embed

class PooledOutput(nn.Module):
    def __init__(self, config, n_output, name="pooled_output"):
        super(PooledOutput, self).__init__()
        self.dense1 = nn.Linear(config.d_model, config.d_model)
        self.dense2 = nn.Linear(config.d_model, n_output, bias=False)

    def forward(self, inputs):
        outputs = F.tanh(self.dense1(inputs))
        outputs = self.dense2(outputs)
        return outputs
    
class PreTrainModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BERT(config)
        self.pooled_output = PooledOutput(config, 2)
        
        # 모델 전체에 가중치 초기화 적용
        self.apply(self._init_weights)

    #  가중치 초기화 함수 정의
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # 선형 계층은 평균 0, 표준편차 0.02의 절단정규분포로 초기화
            nn.init.trunc_normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            # 임베딩 레이어도 동일하게 초기화
            nn.init.trunc_normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm) or isinstance(module, RMSNorm):
            # 정규화 레이어의 가중치는 1로 초기화
            nn.init.ones_(module.weight)
            if hasattr(module, 'bias') and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, enc_tokens, segments):
        logits_cls, logits_lm = self.bert(enc_tokens.long(), segments.long())
        return self.pooled_output(logits_cls), logits_lm

In [13]:
test_model = PreTrainModel(config).to(device)
dummy_enc_tokens = torch.zeros((config.batch_size, config.n_seq), dtype=torch.long).to(device)
dummy_segments = torch.zeros((config.batch_size, config.n_seq), dtype=torch.long).to(device)

summary(
    test_model,
    input_data=[dummy_enc_tokens, dummy_segments],
    col_names=["input_size", "output_size", "num_params"],
    depth=3 # 레이어 중첩을 어디까지 보여줄지 결정합니다
)

Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #
PreTrainModel                                                [64, 128]                 [64, 2]                   --
├─BERT: 1-1                                                  [64, 128]                 [64, 90]                  --
│    └─SharedEmbedding: 2-1                                  [64, 128]                 [64, 128, 90]             720,630
│    └─PositionEmbedding: 2-2                                [64, 128]                 [64, 128, 90]             --
│    │    └─Embedding: 3-1                                   [64, 128]                 [64, 128, 90]             11,520
│    └─Embedding: 2-3                                        [64, 128]                 [64, 128, 90]             180
│    └─LayerNorm: 2-4                                        [64, 128, 90]             [64, 128, 90]             180
│    └─Dropout: 2-5                                     

### Step 6: 학습, 평가 함수 및 최적화 모듈화

In [14]:
def calculate_accuracy(logits, labels, ignore_index=None):
    """
    Logits와 Labels를 비교하여 정확도를 계산하는 함수
    ignore_index가 주어지면 해당 인덱스(예: PAD)는 정확도 계산에서 제외함
    """
    preds = logits.argmax(dim=-1)
    if ignore_index is not None:
        mask = (labels != ignore_index)
        correct = (preds[mask] == labels[mask]).sum().item()
        total = mask.sum().item()
        return correct / total if total > 0 else 0.0
    else:
        correct = (preds == labels).sum().item()
        total = labels.numel()
        return correct / total
    
def lm_loss(y_true, y_pred, scale=1.0):
    """
    loss 계산 함수
    :param y_true: 정답 (bs, n_seq)
    :param y_pred: 예측 값 (bs, n_seq, n_vocab)
    :param scale: Loss에 곱할 가중치
    """
    loss = F.cross_entropy(
        y_pred.view(-1, y_pred.size(-1)), 
        y_true.view(-1), 
        ignore_index=0, 
        label_smoothing=0.1
    )
    return loss * scale

class MultiTaskLoss(nn.Module):
    def __init__(self, dynamic=False, mlm_scale = 1.0):
        """
        :param dynamic: True면 모델이 스스로 가중치를 찾음, False면 mlm_scale 값으로 고정
        :param mlm_scale: dynamic=False일 때 MLM Loss에 곱해줄 고정 스케일 값
        """
        super(MultiTaskLoss, self).__init__()
        self.dynamic = dynamic
        self.mlm_scale = mlm_scale

        if self.dynamic:
            # [핵심 수정] 0으로 초기화하지 않고, mlm_scale을 역산하여 초기 위치 지정
            # NSP는 1.0 배율에서 시작: -ln(1.0) = 0.0
            init_nsp = 0.0
            # MLM은 mlm_scale 배율에서 시작: -ln(mlm_scale)
            init_mlm = -math.log(self.mlm_scale) if self.mlm_scale > 0 else 0.0
            
            # 모델이 이 합리적인 위치(예: 1배 vs 20배)에서부터 최적의 가중치를 미세 조정함
            self.log_vars = nn.Parameter(torch.tensor([init_nsp, init_mlm], dtype=torch.float32))

    def forward(self, loss_nsp, loss_mlm):
        if self.dynamic:
            # 동적 가중치 연산
            precision_nsp = torch.exp(-self.log_vars[0])
            loss0 = precision_nsp * loss_nsp + 0.5 * self.log_vars[0]

            precision_mlm = torch.exp(-self.log_vars[1])
            loss1 = precision_mlm * loss_mlm + 0.5 * self.log_vars[1]

            return loss0 + loss1
        else:
            # 정적 스케일 고정 연산 (loss_nsp는 1배, loss_mlm은 지정된 스케일 배수)
            return loss_nsp + (loss_mlm * self.mlm_scale)
        
class CosineSchedule:
    def __init__(self, optimizer=None, train_steps=4000, warmup_steps=500, max_lr=2.5e-4):
        self.optimizer = optimizer
        self.train_steps = train_steps
        self.warmup_steps = warmup_steps
        self.max_lr = max_lr
        self.step_num = 0  # 변수명 통일

    def get_lr(self):
        if self.step_num <= self.warmup_steps:
            # 워밍업 구간: 0부터 max_lr까지 선형적으로 증가
            lr = (self.step_num / self.warmup_steps) * self.max_lr
        else:
            # 코사인 감소 구간
            progress = (self.step_num - self.warmup_steps) / max(1, self.train_steps - self.warmup_steps)
            lr = 0.5 * self.max_lr * (1 + math.cos(math.pi * progress))
        return lr

    def step(self):
        self.step_num += 1  # current_step에서 step_num으로 버그 수정
        lr = self.get_lr()
        
        # 클래스 내부에서 직접 optimizer의 lr을 업데이트
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr

def print_learned_weights(mtl_loss_module):
    # log_vars[0]은 NSP, log_vars[1]은 MLM
    log_vars = mtl_loss_module.log_vars.detach().cpu()
    
    # 지수 변환을 통해 실제 곱해지는 가중치(Precision) 계산
    # 수식: weight = exp(-log_var)
    weights = torch.exp(-log_vars)
    
    nsp_weight = weights[0].item()
    mlm_weight = weights[1].item()

    print("-" * 30)
    print(f"[최종 학습된 가중치 지표]")
    print(f" - NSP 가중치: {nsp_weight:.4f}")
    print(f" - MLM 가중치: {mlm_weight:.4f}")
    print(f" - MLM이 NSP보다 {mlm_weight / nsp_weight:.2f}배 더 중요하게 학습됨")
    print("-" * 30)

In [15]:
def validate(model, val_dataloader, device):
    """모델 평가(Validation)를 수행하고 메트릭을 반환하는 함수"""
    model.eval()
    val_total_loss, val_nsp_loss, val_mlm_loss, val_nsp_acc, val_mlm_acc = 0, 0, 0, 0, 0
    
    all_nsp_preds = []
    all_nsp_labels = []
    all_nsp_probs = []
    
    with torch.no_grad():
        for batch in val_dataloader:
            enc_tokens, segments, labels_nsp, labels_mlm = [b.to(device).long() for b in batch]
            
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits_nsp, logits_mlm = model(enc_tokens, segments)
                
                loss_nsp = F.cross_entropy(logits_nsp, labels_nsp)
                loss_mlm = lm_loss(labels_mlm, logits_mlm, scale=1.0) 
                loss = loss_nsp + loss_mlm
            
            val_total_loss += loss.item() 
            val_nsp_loss += loss_nsp.item()
            val_mlm_loss += loss_mlm.item() 
            val_nsp_acc += calculate_accuracy(logits_nsp, labels_nsp)
            val_mlm_acc += calculate_accuracy(logits_mlm, labels_mlm, ignore_index=0)
            
            nsp_preds_class = logits_nsp.argmax(dim=-1)
            nsp_probs = torch.softmax(logits_nsp, dim=-1)[:, 1] 
            all_nsp_preds.extend(nsp_preds_class.cpu().tolist())
            all_nsp_labels.extend(labels_nsp.cpu().tolist())
            all_nsp_probs.extend(nsp_probs.cpu().tolist())
            
    num_batches = len(val_dataloader)
    
    metrics = {
        'val_total_loss': val_total_loss / num_batches,
        'val_nsp_loss': val_nsp_loss / num_batches,
        'val_mlm_loss': val_mlm_loss / num_batches,
        'val_nsp_acc': val_nsp_acc / num_batches,
        'val_mlm_acc': val_mlm_acc / num_batches,
        'val_mlm_ppl': math.exp(val_mlm_loss / num_batches), 
        'val_nsp_f1': f1_score(all_nsp_labels, all_nsp_preds, zero_division=0)
    } 
    
    try:
        metrics['val_nsp_roc_auc'] = roc_auc_score(all_nsp_labels, all_nsp_probs)
    except ValueError:
        metrics['val_nsp_roc_auc'] = 0.5 
        
    return metrics

In [16]:
# 3. 최적화가 적용된 모델 학습 함수
def train(model, train_loader, val_loader, config, device, model_name='v1', mlm_scale = 20.0, use_dynamic_loss = False,
          lr=2e-4, accumulation_steps=4):
    model.to(device)
    mtl_loss_module = MultiTaskLoss(dynamic=use_dynamic_loss, mlm_scale=mlm_scale).to(device)

    optimizer_params = list(model.parameters())
    if use_dynamic_loss:
        optimizer_params += list(mtl_loss_module.parameters())

    # [최적화 1] optim.Adam 대신 optim.AdamW 사용 (Weight Decay 적용)
    # 1. 파라미터 이름(name)을 확인해서 정규화/편향 관련 가중치인지 분류합니다.
    no_decay = ['bias', 'norm', 'LayerNorm', 'RMSNorm']
    optimizer_grouped_parameters = [
        # 일반 가중치 행렬: weight_decay 0.01 적용
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        # 편향 및 정규화 계층: weight_decay 0.0 (미적용)
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]
    
    # 동적 손실 파라미터도 정규화에서 제외합니다.
    if use_dynamic_loss:
        optimizer_grouped_parameters.append({'params': mtl_loss_module.parameters(), 'weight_decay': 0.0})

    # 분류된 파라미터 그룹을 AdamW에 전달
    optimizer = optim.AdamW(optimizer_grouped_parameters, lr=lr)

    # [최적화 2 & 3] Accumulation Step을 반영하여 실제 Optimizer가 업데이트되는 횟수 계산
    train_steps = math.ceil(len(train_loader) / accumulation_steps) * config.epochs
    scheduler = CosineSchedule(optimizer=optimizer, train_steps=train_steps, warmup_steps=max(100, train_steps * 0.1), max_lr=lr)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    save_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

    history = collections.defaultdict(list)

    for epoch in range(config.epochs):
        model.train()
        if use_dynamic_loss:
            mtl_loss_module.train()

        total_loss, total_nsp_loss, total_mlm_loss, total_nsp_acc, total_mlm_acc = 0, 0, 0, 0, 0
        optimizer.zero_grad() # 에포크 시작 시 기울기 초기화
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Train]")
        for i, batch in enumerate(progress_bar):
            enc_tokens, segments, labels_nsp, labels_mlm = [b.to(device).long() for b in batch]

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits_nsp, logits_mlm = model(enc_tokens, segments)                
                loss_nsp = F.cross_entropy(logits_nsp, labels_nsp)    
                raw_loss_mlm = lm_loss(labels_mlm, logits_mlm, scale=1.0)
                
                # 역전파용 최종 Loss (가중치 적용됨)
                loss = mtl_loss_module(loss_nsp, raw_loss_mlm)
                scaled_loss = loss / accumulation_steps

            scaler.scale(scaled_loss).backward()
            
            if use_dynamic_loss:
                # 동적 모드: exp(-log_var) 값을 가져와서 곱함
                w_nsp_val = torch.exp(-mtl_loss_module.log_vars[0]).item() * loss_nsp.item()
                w_mlm_val = torch.exp(-mtl_loss_module.log_vars[1]).item() * raw_loss_mlm.item()
            else:
                # 정적 모드: nsp는 1배, mlm은 지정된 mlm_scale 배
                w_nsp_val = loss_nsp.item()
                w_mlm_val = raw_loss_mlm.item() * mlm_scale

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

            # 누적 및 출력 (이제 가중치가 반영된 값을 사용)
            total_loss += loss.item()
            total_nsp_loss += w_nsp_val  # 가중치 반영된 NSP 지분
            total_mlm_loss += w_mlm_val  # 가중치 반영된 MLM 지분
            
            nsp_acc = calculate_accuracy(logits_nsp, labels_nsp)
            mlm_acc = calculate_accuracy(logits_mlm, labels_mlm, ignore_index=0)
            total_nsp_acc += nsp_acc
            total_mlm_acc += mlm_acc

            # 화면 출력: 이제 T_Loss = N_Loss + M_Loss (+ 동적일 땐 상수항) 의 관계가 눈에 보임
            progress_bar.set_postfix({
                "T_Loss": f"{loss.item():.4f}", 
                "WN_Loss": f"{w_nsp_val:.4f}",  # Weighted NSP
                "WM_Loss": f"{w_mlm_val:.4f}",  # Weighted MLM
                "N_Acc": f"{nsp_acc:.4f}", 
                "M_Acc": f"{mlm_acc:.4f}"
            })

        history['total_loss'].append(total_loss / len(train_loader))
        history['nsp_loss'].append(total_nsp_loss / len(train_loader))
        history['mlm_loss'].append(total_mlm_loss / len(train_loader))
        history['nsp_acc'].append(total_nsp_acc / len(train_loader))
        history['mlm_acc'].append(total_mlm_acc / len(train_loader))

        # --- Validation Phase ---
        val_metrics = validate(model, val_loader, device)
        
        # 반환받은 메트릭을 history에 기록
        for key, value in val_metrics.items():
            history[key].append(value)

        # --- 로그 출력 ---
        print(f"Epoch {epoch+1} Summary:")
        print(f" [Train] Total Loss: {history['total_loss'][-1]:.4f} | NSP Loss: {history['nsp_loss'][-1]:.4f} | MLM Loss: {history['mlm_loss'][-1]:.4f} | NSP Acc: {history['nsp_acc'][-1]:.4f} | MLM Acc: {history['mlm_acc'][-1]:.4f}")
        print(f" [Val]   Total Loss: {history['val_total_loss'][-1]:.4f} | NSP Loss: {history['val_nsp_loss'][-1]:.4f} | MLM Loss: {history['val_mlm_loss'][-1]:.4f} | NSP Acc: {history['val_nsp_acc'][-1]:.4f} | MLM Acc: {history['val_mlm_acc'][-1]:.4f}")
        print(f" [Val Metrics] NSP F1: {history['val_nsp_f1'][-1]:.4f} | NSP AUC: {history['val_nsp_roc_auc'][-1]:.4f} | MLM PPL: {history['val_mlm_ppl'][-1]:.2f}\n")
        
        # --- 모델 저장 ---
        if (epoch + 1) == config.epochs: 
            torch.save(model.state_dict(), save_path)

        if (epoch + 1) == config.epochs and use_dynamic_loss:
            print_learned_weights(mtl_loss_module)
            
    return dict(history)

### v1: MLM scale 20

In [ ]:
%time
model_v1 = PreTrainModel(config).to(device)
print("=== Training v1 (MLM Loss Scale: 20.0) ===")
history_v1 = train(model_v1, train_loader, val_loader, config, device, model_name='v1', accumulation_steps=1)

CPU times: user 4 μs, sys: 0 ns, total: 4 μs
Wall time: 7.63 μs
=== Training v1 (MLM Loss Scale: 20.0) ===


Epoch 1/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.10it/s, T_Loss=145.7725, WN_Loss=0.5569, WM_Loss=145.2156, N_Acc=0.6250, M_Acc=0.0917]


Epoch 1 Summary:
 [Train] Total Loss: 155.9251 | NSP Loss: 0.6543 | MLM Loss: 155.2708 | NSP Acc: 0.5624 | MLM Acc: 0.0739
 [Val]   Total Loss: 7.8741 | NSP Loss: 0.5847 | MLM Loss: 7.2895 | NSP Acc: 0.6361 | MLM Acc: 0.0948
 [Val Metrics] NSP F1: 0.6658 | NSP AUC: 0.7283 | MLM PPL: 1464.80



Epoch 2/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.15it/s, T_Loss=137.8925, WN_Loss=0.5830, WM_Loss=137.3095, N_Acc=0.6406, M_Acc=0.1574]


Epoch 2 Summary:
 [Train] Total Loss: 141.9749 | NSP Loss: 0.5592 | MLM Loss: 141.4157 | NSP Acc: 0.6545 | MLM Acc: 0.1277
 [Val]   Total Loss: 7.3746 | NSP Loss: 0.5325 | MLM Loss: 6.8421 | NSP Acc: 0.6700 | MLM Acc: 0.1561
 [Val Metrics] NSP F1: 0.6354 | NSP AUC: 0.7721 | MLM PPL: 936.48



...

Epoch 8 Summary:
 [Train] Total Loss: 111.5739 | NSP Loss: 0.5184 | MLM Loss: 111.0555 | NSP Acc: 0.6701 | MLM Acc: 0.2286
 [Val]   Total Loss: 5.9923 | NSP Loss: 0.5102 | MLM Loss: 5.4821 | NSP Acc: 0.6758 | MLM Acc: 0.2363
 [Val Metrics] NSP F1: 0.7233 | NSP AUC: 0.7817 | MLM PPL: 240.34



Epoch 9/10 [Train]: 100%|██████████| 5000/5000 [03:10<00:00, 26.18it/s, T_Loss=113.6654, WN_Loss=0.5366, WM_Loss=113.1288, N_Acc=0.6094, M_Acc=0.2305]


Epoch 9 Summary:
 [Train] Total Loss: 111.0287 | NSP Loss: 0.5163 | MLM Loss: 110.5124 | NSP Acc: 0.6722 | MLM Acc: 0.2307
 [Val]   Total Loss: 5.9788 | NSP Loss: 0.5099 | MLM Loss: 5.4689 | NSP Acc: 0.6760 | MLM Acc: 0.2376
 [Val Metrics] NSP F1: 0.7133 | NSP AUC: 0.7822 | MLM PPL: 237.20



Epoch 10/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.15it/s, T_Loss=111.2217, WN_Loss=0.5664, WM_Loss=110.6553, N_Acc=0.5469, M_Acc=0.2337]


Epoch 10 Summary:
 [Train] Total Loss: 110.8217 | NSP Loss: 0.5157 | MLM Loss: 110.3060 | NSP Acc: 0.6718 | MLM Acc: 0.2315
 [Val]   Total Loss: 5.9763 | NSP Loss: 0.5095 | MLM Loss: 5.4668 | NSP Acc: 0.6761 | MLM Acc: 0.2377
 [Val Metrics] NSP F1: 0.7122 | NSP AUC: 0.7826 | MLM PPL: 236.70



### 개선 방안 
### v2: MLM scale 1

In [20]:
%time
model_v2 = PreTrainModel(config).to(device)
print("\n=== Training v2 (MLM Loss Scale: 1.0)===")
history_v2 = train(model_v2, train_loader, val_loader, config, device, model_name='v2', accumulation_steps=1, mlm_scale=1.0)

CPU times: user 0 ns, sys: 4 μs, total: 4 μs
Wall time: 9.06 μs

=== Training v2 (MLM Loss Scale: 1.0)===


Epoch 1/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.16it/s, T_Loss=7.9074, WN_Loss=0.5474, WM_Loss=7.3600, N_Acc=0.5625, M_Acc=0.0862]


Epoch 1 Summary:
 [Train] Total Loss: 8.3937 | NSP Loss: 0.5897 | MLM Loss: 7.8040 | NSP Acc: 0.6184 | MLM Acc: 0.0730
 [Val]   Total Loss: 7.9115 | NSP Loss: 0.5441 | MLM Loss: 7.3674 | NSP Acc: 0.6484 | MLM Acc: 0.0853
 [Val Metrics] NSP F1: 0.7099 | NSP AUC: 0.7477 | MLM PPL: 1583.58



Epoch 2/10 [Train]: 100%|██████████| 5000/5000 [03:10<00:00, 26.21it/s, T_Loss=7.5871, WN_Loss=0.6112, WM_Loss=6.9758, N_Acc=0.6719, M_Acc=0.1612]


Epoch 2 Summary:
 [Train] Total Loss: 7.7104 | NSP Loss: 0.5264 | MLM Loss: 7.1839 | NSP Acc: 0.6659 | MLM Acc: 0.1186
 [Val]   Total Loss: 7.4648 | NSP Loss: 0.5133 | MLM Loss: 6.9515 | NSP Acc: 0.6787 | MLM Acc: 0.1493
 [Val Metrics] NSP F1: 0.7034 | NSP AUC: 0.7838 | MLM PPL: 1044.69



...

Epoch 8 Summary:
 [Train] Total Loss: 6.3495 | NSP Loss: 0.4593 | MLM Loss: 5.8902 | NSP Acc: 0.7098 | MLM Acc: 0.2052
 [Val]   Total Loss: 6.3096 | NSP Loss: 0.4955 | MLM Loss: 5.8142 | NSP Acc: 0.6857 | MLM Acc: 0.2119
 [Val Metrics] NSP F1: 0.7201 | NSP AUC: 0.7983 | MLM PPL: 335.02



Epoch 9/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.18it/s, T_Loss=5.9638, WN_Loss=0.3515, WM_Loss=5.6122, N_Acc=0.7812, M_Acc=0.2340]


Epoch 9 Summary:
 [Train] Total Loss: 6.3115 | NSP Loss: 0.4547 | MLM Loss: 5.8568 | NSP Acc: 0.7135 | MLM Acc: 0.2071
 [Val]   Total Loss: 6.2913 | NSP Loss: 0.4953 | MLM Loss: 5.7960 | NSP Acc: 0.6859 | MLM Acc: 0.2130
 [Val Metrics] NSP F1: 0.6919 | NSP AUC: 0.7979 | MLM PPL: 328.99



Epoch 10/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.18it/s, T_Loss=6.4527, WN_Loss=0.4478, WM_Loss=6.0049, N_Acc=0.7500, M_Acc=0.1858]


Epoch 10 Summary:
 [Train] Total Loss: 6.2968 | NSP Loss: 0.4518 | MLM Loss: 5.8450 | NSP Acc: 0.7157 | MLM Acc: 0.2077
 [Val]   Total Loss: 6.2911 | NSP Loss: 0.4975 | MLM Loss: 5.7937 | NSP Acc: 0.6853 | MLM Acc: 0.2132
 [Val Metrics] NSP F1: 0.6854 | NSP AUC: 0.7978 | MLM PPL: 328.21



### v3: MLM scale 10

In [21]:
%time
model_v3 = PreTrainModel(config).to(device)
print("\n=== Training v3 (MLM Loss Scale: 10.0) ===")
history_v3 = train(model_v3, train_loader, val_loader, config, device, model_name='v3', accumulation_steps=1, mlm_scale=10.0)

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 10 μs

=== Training v3 (MLM Loss Scale: 10.0) ===


Epoch 1/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.17it/s, T_Loss=73.7796, WN_Loss=0.6036, WM_Loss=73.1760, N_Acc=0.5625, M_Acc=0.0954]


Epoch 1 Summary:
 [Train] Total Loss: 78.3480 | NSP Loss: 0.6346 | MLM Loss: 77.7134 | NSP Acc: 0.5757 | MLM Acc: 0.0737
 [Val]   Total Loss: 7.8482 | NSP Loss: 0.5508 | MLM Loss: 7.2974 | NSP Acc: 0.6477 | MLM Acc: 0.0958
 [Val Metrics] NSP F1: 0.6583 | NSP AUC: 0.7419 | MLM PPL: 1476.48



Epoch 2/10 [Train]: 100%|██████████| 5000/5000 [03:10<00:00, 26.19it/s, T_Loss=70.4513, WN_Loss=0.5455, WM_Loss=69.9058, N_Acc=0.6250, M_Acc=0.1335]


Epoch 2 Summary:
 [Train] Total Loss: 71.2762 | NSP Loss: 0.5377 | MLM Loss: 70.7386 | NSP Acc: 0.6604 | MLM Acc: 0.1281
 [Val]   Total Loss: 7.3569 | NSP Loss: 0.5228 | MLM Loss: 6.8341 | NSP Acc: 0.6700 | MLM Acc: 0.1572
 [Val Metrics] NSP F1: 0.6668 | NSP AUC: 0.7738 | MLM PPL: 929.00



...

Epoch 8 Summary:
 [Train] Total Loss: 56.4832 | NSP Loss: 0.5046 | MLM Loss: 55.9786 | NSP Acc: 0.6779 | MLM Acc: 0.2247
 [Val]   Total Loss: 6.0233 | NSP Loss: 0.5019 | MLM Loss: 5.5214 | NSP Acc: 0.6824 | MLM Acc: 0.2324
 [Val Metrics] NSP F1: 0.6663 | NSP AUC: 0.7904 | MLM PPL: 249.97



Epoch 9/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.17it/s, T_Loss=56.9940, WN_Loss=0.6389, WM_Loss=56.3551, N_Acc=0.6406, M_Acc=0.2106]


Epoch 9 Summary:
 [Train] Total Loss: 56.2046 | NSP Loss: 0.5028 | MLM Loss: 55.7018 | NSP Acc: 0.6783 | MLM Acc: 0.2265
 [Val]   Total Loss: 6.0079 | NSP Loss: 0.5000 | MLM Loss: 5.5079 | NSP Acc: 0.6820 | MLM Acc: 0.2334
 [Val Metrics] NSP F1: 0.6697 | NSP AUC: 0.7907 | MLM PPL: 246.64



Epoch 10/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.17it/s, T_Loss=57.1984, WN_Loss=0.5538, WM_Loss=56.6447, N_Acc=0.7188, M_Acc=0.2239]


Epoch 10 Summary:
 [Train] Total Loss: 56.1005 | NSP Loss: 0.5019 | MLM Loss: 55.5986 | NSP Acc: 0.6790 | MLM Acc: 0.2273
 [Val]   Total Loss: 6.0061 | NSP Loss: 0.5003 | MLM Loss: 5.5058 | NSP Acc: 0.6827 | MLM Acc: 0.2338
 [Val Metrics] NSP F1: 0.6749 | NSP AUC: 0.7907 | MLM PPL: 246.11



### v4: MLM scale 50

In [33]:
%time
model_v4 = PreTrainModel(config).to(device)
print("\n=== Training v4 ((MLM Loss Scale: 50.0) ===")
history_v4 = train(model_v4, train_loader, val_loader, config, device, model_name='v4', accumulation_steps=1, mlm_scale=50.0)

CPU times: user 3 μs, sys: 0 ns, total: 3 μs
Wall time: 6.44 μs

=== Training v4 ((MLM Loss Scale: 50.0) ===


Epoch 1/10 [Train]: 100%|██████████| 5000/5000 [03:12<00:00, 26.03it/s, T_Loss=365.7494, WN_Loss=0.5835, WM_Loss=365.1659, N_Acc=0.6406, M_Acc=0.0945]


Epoch 1 Summary:
 [Train] Total Loss: 388.9827 | NSP Loss: 0.6606 | MLM Loss: 388.3221 | NSP Acc: 0.5636 | MLM Acc: 0.0744
 [Val]   Total Loss: 7.9024 | NSP Loss: 0.6096 | MLM Loss: 7.2928 | NSP Acc: 0.6313 | MLM Acc: 0.0962
 [Val Metrics] NSP F1: 0.7000 | NSP AUC: 0.7202 | MLM PPL: 1469.66



Epoch 2/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.06it/s, T_Loss=339.1007, WN_Loss=0.6021, WM_Loss=338.4986, N_Acc=0.6719, M_Acc=0.1565]


Epoch 2 Summary:
 [Train] Total Loss: 354.1327 | NSP Loss: 0.6022 | MLM Loss: 353.5305 | NSP Acc: 0.6338 | MLM Acc: 0.1285
 [Val]   Total Loss: 7.4270 | NSP Loss: 0.5879 | MLM Loss: 6.8391 | NSP Acc: 0.6345 | MLM Acc: 0.1557
 [Val Metrics] NSP F1: 0.6997 | NSP AUC: 0.7312 | MLM PPL: 933.66



...

Epoch 8 Summary:
 [Train] Total Loss: 282.2739 | NSP Loss: 0.5566 | MLM Loss: 281.7173 | NSP Acc: 0.6463 | MLM Acc: 0.2225
 [Val]   Total Loss: 6.1107 | NSP Loss: 0.5505 | MLM Loss: 5.5602 | NSP Acc: 0.6526 | MLM Acc: 0.2297
 [Val Metrics] NSP F1: 0.6790 | NSP AUC: 0.7484 | MLM PPL: 259.88



Epoch 9/10 [Train]: 100%|██████████| 5000/5000 [03:12<00:00, 25.99it/s, T_Loss=283.7260, WN_Loss=0.6303, WM_Loss=283.0957, N_Acc=0.5469, M_Acc=0.2050]


Epoch 9 Summary:
 [Train] Total Loss: 280.7921 | NSP Loss: 0.5545 | MLM Loss: 280.2376 | NSP Acc: 0.6480 | MLM Acc: 0.2245
 [Val]   Total Loss: 6.0963 | NSP Loss: 0.5512 | MLM Loss: 5.5451 | NSP Acc: 0.6517 | MLM Acc: 0.2310
 [Val Metrics] NSP F1: 0.6560 | NSP AUC: 0.7487 | MLM PPL: 255.97



Epoch 10/10 [Train]: 100%|██████████| 5000/5000 [03:11<00:00, 26.08it/s, T_Loss=280.6855, WN_Loss=0.5377, WM_Loss=280.1478, N_Acc=0.6562, M_Acc=0.2213]


Epoch 10 Summary:
 [Train] Total Loss: 280.2247 | NSP Loss: 0.5538 | MLM Loss: 279.6709 | NSP Acc: 0.6488 | MLM Acc: 0.2255
 [Val]   Total Loss: 6.0937 | NSP Loss: 0.5508 | MLM Loss: 5.5428 | NSP Acc: 0.6528 | MLM Acc: 0.2313
 [Val Metrics] NSP F1: 0.6655 | NSP AUC: 0.7488 | MLM PPL: 255.40



### v5: MLM scale 1 + dynamic

In [34]:
%time
model_v5 = PreTrainModel(config).to(device)
print("\n=== Training v5 (Dynamic MLM Loss) ===")
history_v5 = train(model_v5, train_loader, val_loader, config, device, model_name='v5', accumulation_steps=1, mlm_scale=1.0, use_dynamic_loss=True)

CPU times: user 3 μs, sys: 0 ns, total: 3 μs
Wall time: 7.39 μs

=== Training v5 (Dynamic MLM Loss) ===


Epoch 1/10 [Train]: 100%|██████████| 5000/5000 [03:14<00:00, 25.75it/s, T_Loss=6.4125, WN_Loss=0.6280, WM_Loss=5.7982, N_Acc=0.6875, M_Acc=0.0984]


Epoch 1 Summary:
 [Train] Total Loss: 7.9084 | NSP Loss: 0.6356 | MLM Loss: 7.2809 | NSP Acc: 0.6238 | MLM Acc: 0.0712
 [Val]   Total Loss: 7.9504 | NSP Loss: 0.5408 | MLM Loss: 7.4096 | NSP Acc: 0.6566 | MLM Acc: 0.0846
 [Val Metrics] NSP F1: 0.6808 | NSP AUC: 0.7542 | MLM PPL: 1651.74



Epoch 2/10 [Train]: 100%|██████████| 5000/5000 [03:14<00:00, 25.77it/s, T_Loss=4.0401, WN_Loss=0.8430, WM_Loss=3.0154, N_Acc=0.7344, M_Acc=0.1286]


Epoch 2 Summary:
 [Train] Total Loss: 5.2977 | NSP Loss: 0.8360 | MLM Loss: 4.4253 | NSP Acc: 0.6732 | MLM Acc: 0.1022
 [Val]   Total Loss: 7.6406 | NSP Loss: 0.5119 | MLM Loss: 7.1286 | NSP Acc: 0.6834 | MLM Acc: 0.1304
 [Val Metrics] NSP F1: 0.6667 | NSP AUC: 0.7926 | MLM PPL: 1247.18



...

Epoch 8 Summary:
 [Train] Total Loss: 3.0412 | NSP Loss: 0.9838 | MLM Loss: 0.9999 | NSP Acc: 0.7558 | MLM Acc: 0.1609
 [Val]   Total Loss: 7.3856 | NSP Loss: 0.5661 | MLM Loss: 6.8195 | NSP Acc: 0.6830 | MLM Acc: 0.1639
 [Val Metrics] NSP F1: 0.6882 | NSP AUC: 0.7950 | MLM PPL: 915.55



Epoch 9/10 [Train]: 100%|██████████| 5000/5000 [03:13<00:00, 25.89it/s, T_Loss=3.1228, WN_Loss=1.0883, WM_Loss=1.0025, N_Acc=0.7500, M_Acc=0.1327]


Epoch 9 Summary:
 [Train] Total Loss: 3.0126 | NSP Loss: 0.9749 | MLM Loss: 1.0000 | NSP Acc: 0.7660 | MLM Acc: 0.1612
 [Val]   Total Loss: 7.4148 | NSP Loss: 0.5949 | MLM Loss: 6.8198 | NSP Acc: 0.6814 | MLM Acc: 0.1638
 [Val Metrics] NSP F1: 0.6905 | NSP AUC: 0.7937 | MLM PPL: 915.84



Epoch 10/10 [Train]: 100%|██████████| 5000/5000 [03:12<00:00, 25.92it/s, T_Loss=2.9171, WN_Loss=0.9069, WM_Loss=0.9815, N_Acc=0.7500, M_Acc=0.1790]


Epoch 10 Summary:
 [Train] Total Loss: 2.9950 | NSP Loss: 0.9656 | MLM Loss: 0.9998 | NSP Acc: 0.7726 | MLM Acc: 0.1612
 [Val]   Total Loss: 7.4260 | NSP Loss: 0.6064 | MLM Loss: 6.8195 | NSP Acc: 0.6807 | MLM Acc: 0.1640
 [Val Metrics] NSP F1: 0.6907 | NSP AUC: 0.7931 | MLM PPL: 915.55

------------------------------
[최종 학습된 가중치 지표]
 - NSP 가중치: 2.4401
 - MLM 가중치: 0.1465
 - MLM이 NSP보다 0.06배 더 중요하게 학습됨
------------------------------


### 개선 방안 
### v6: Gradient Accumulation 적용 (256 [batch size 64 * 4])

In [80]:
%time
model_v6 = PreTrainModel(config).to(device)
print("\n=== Training v6 (Gradient Accumulation: batch size * 4 ) ===")
history_v6 = train(model_v6, train_loader, val_loader, config, device, model_name='v6', accumulation_steps=4, mlm_scale=1.0)

CPU times: user 3 μs, sys: 0 ns, total: 3 μs
Wall time: 8.11 μs

=== Training v6 (Gradient Accumulation: batch size * 4 ) ===


Epoch 1/10 [Train]: 100%|██████████| 5000/5000 [03:04<00:00, 27.16it/s, T_Loss=7.9589, WN_Loss=0.4981, WM_Loss=7.4609, N_Acc=0.7188, M_Acc=0.0906]


Epoch 1 Summary:
 [Train] Total Loss: 8.5095 | NSP Loss: 0.5965 | MLM Loss: 7.9129 | NSP Acc: 0.6143 | MLM Acc: 0.0692
 [Val]   Total Loss: 7.9695 | NSP Loss: 0.5358 | MLM Loss: 7.4337 | NSP Acc: 0.6514 | MLM Acc: 0.0823
 [Val Metrics] NSP F1: 0.7129 | NSP AUC: 0.7564 | MLM PPL: 1692.03



Epoch 2/10 [Train]: 100%|██████████| 5000/5000 [03:04<00:00, 27.13it/s, T_Loss=7.7013, WN_Loss=0.4572, WM_Loss=7.2442, N_Acc=0.7500, M_Acc=0.1082]


Epoch 2 Summary:
 [Train] Total Loss: 7.8224 | NSP Loss: 0.5130 | MLM Loss: 7.3094 | NSP Acc: 0.6756 | MLM Acc: 0.0978
 [Val]   Total Loss: 7.6737 | NSP Loss: 0.5098 | MLM Loss: 7.1639 | NSP Acc: 0.6783 | MLM Acc: 0.1245
 [Val Metrics] NSP F1: 0.7458 | NSP AUC: 0.7922 | MLM PPL: 1291.92



...

Epoch 8 Summary:
 [Train] Total Loss: 7.1599 | NSP Loss: 0.4586 | MLM Loss: 6.7013 | NSP Acc: 0.7092 | MLM Acc: 0.1660
 [Val]   Total Loss: 7.2073 | NSP Loss: 0.4969 | MLM Loss: 6.7104 | NSP Acc: 0.6900 | MLM Acc: 0.1664
 [Val Metrics] NSP F1: 0.6960 | NSP AUC: 0.7994 | MLM PPL: 820.91



Epoch 9/10 [Train]: 100%|██████████| 5000/5000 [03:04<00:00, 27.16it/s, T_Loss=7.1089, WN_Loss=0.4614, WM_Loss=6.6475, N_Acc=0.7188, M_Acc=0.1630]


Epoch 9 Summary:
 [Train] Total Loss: 7.1451 | NSP Loss: 0.4563 | MLM Loss: 6.6888 | NSP Acc: 0.7108 | MLM Acc: 0.1666
 [Val]   Total Loss: 7.2032 | NSP Loss: 0.4994 | MLM Loss: 6.7038 | NSP Acc: 0.6880 | MLM Acc: 0.1669
 [Val Metrics] NSP F1: 0.7023 | NSP AUC: 0.7984 | MLM PPL: 815.53



Epoch 10/10 [Train]: 100%|██████████| 5000/5000 [03:04<00:00, 27.13it/s, T_Loss=6.9572, WN_Loss=0.3862, WM_Loss=6.5710, N_Acc=0.7969, M_Acc=0.1714]


Epoch 10 Summary:
 [Train] Total Loss: 7.1396 | NSP Loss: 0.4552 | MLM Loss: 6.6845 | NSP Acc: 0.7128 | MLM Acc: 0.1667
 [Val]   Total Loss: 7.2050 | NSP Loss: 0.5022 | MLM Loss: 6.7028 | NSP Acc: 0.6885 | MLM Acc: 0.1669
 [Val Metrics] NSP F1: 0.6971 | NSP AUC: 0.7989 | MLM PPL: 814.69



### 개선 방안 
### v7: Training Data Size Variation (355,555 [5000 steps] -> 128,000 [1800 steps])

In [81]:
#전체 데이터 수 3957761
# batch size: 64, train: val > 9:1
# 355555: 5000step, 128000: 1800step
train_loader, val_loader, vocab = load_datasets(PRETRAIN_JSON_PATH, config, count=128000)

100%|██████████| 128000/128000 [00:17<00:00, 7185.94it/s]


data load early stop 128000 128000


In [82]:
%time
model_v7 = PreTrainModel(config).to(device)
print("\n=== Training v7 data: ===")
history_v7 = train(model_v7, train_loader, val_loader, config, device, model_name='v7', accumulation_steps=1, mlm_scale=1.0)

CPU times: user 3 μs, sys: 1e+03 ns, total: 4 μs
Wall time: 7.63 μs

=== Training v7 data: ===


Epoch 1/10 [Train]: 100%|██████████| 1800/1800 [01:08<00:00, 26.10it/s, T_Loss=8.1081, WN_Loss=0.6259, WM_Loss=7.4822, N_Acc=0.6250, M_Acc=0.0848]


Epoch 1 Summary:
 [Train] Total Loss: 8.5064 | NSP Loss: 0.6362 | MLM Loss: 7.8702 | NSP Acc: 0.5750 | MLM Acc: 0.0708
 [Val]   Total Loss: 8.0434 | NSP Loss: 0.5928 | MLM Loss: 7.4505 | NSP Acc: 0.5904 | MLM Acc: 0.0848
 [Val Metrics] NSP F1: 0.6983 | NSP AUC: 0.6782 | MLM PPL: 1720.78



Epoch 2/10 [Train]: 100%|██████████| 1800/1800 [01:08<00:00, 26.09it/s, T_Loss=7.8352, WN_Loss=0.5777, WM_Loss=7.2574, N_Acc=0.7188, M_Acc=0.1051]


Epoch 2 Summary:
 [Train] Total Loss: 7.8994 | NSP Loss: 0.5889 | MLM Loss: 7.3105 | NSP Acc: 0.6041 | MLM Acc: 0.0988
 [Val]   Total Loss: 7.7579 | NSP Loss: 0.5847 | MLM Loss: 7.1732 | NSP Acc: 0.6113 | MLM Acc: 0.1216
 [Val Metrics] NSP F1: 0.4291 | NSP AUC: 0.7010 | MLM PPL: 1304.06



...

Epoch 8 Summary:
 [Train] Total Loss: 7.2399 | NSP Loss: 0.4921 | MLM Loss: 6.7478 | NSP Acc: 0.7126 | MLM Acc: 0.1649
 [Val]   Total Loss: 7.4579 | NSP Loss: 0.6472 | MLM Loss: 6.8107 | NSP Acc: 0.6082 | MLM Acc: 0.1622
 [Val Metrics] NSP F1: 0.6093 | NSP AUC: 0.6962 | MLM PPL: 907.54



Epoch 9/10 [Train]: 100%|██████████| 1800/1800 [01:08<00:00, 26.11it/s, T_Loss=7.3077, WN_Loss=0.5669, WM_Loss=6.7408, N_Acc=0.6875, M_Acc=0.1642]


Epoch 9 Summary:
 [Train] Total Loss: 7.2218 | NSP Loss: 0.4820 | MLM Loss: 6.7398 | NSP Acc: 0.7209 | MLM Acc: 0.1652
 [Val]   Total Loss: 7.4715 | NSP Loss: 0.6639 | MLM Loss: 6.8076 | NSP Acc: 0.6072 | MLM Acc: 0.1621
 [Val Metrics] NSP F1: 0.6139 | NSP AUC: 0.6949 | MLM PPL: 904.73



Epoch 10/10 [Train]: 100%|██████████| 1800/1800 [01:08<00:00, 26.13it/s, T_Loss=7.0932, WN_Loss=0.4515, WM_Loss=6.6417, N_Acc=0.7656, M_Acc=0.1872]


Epoch 10 Summary:
 [Train] Total Loss: 7.2119 | NSP Loss: 0.4753 | MLM Loss: 6.7366 | NSP Acc: 0.7257 | MLM Acc: 0.1654
 [Val]   Total Loss: 7.4845 | NSP Loss: 0.6767 | MLM Loss: 6.8079 | NSP Acc: 0.6080 | MLM Acc: 0.1622
 [Val Metrics] NSP F1: 0.6133 | NSP AUC: 0.6953 | MLM PPL: 904.94



### Step 7: 비교 시각화

In [36]:
def compare_imshow(histories_dict):
    """
    많은 모델 히스코그램을 한 그래프에서 겹쳐 그립니다.
    """
    # 텐서 값들을 float로 변환
    safe_histories = {}
    for name, history in histories_dict.items():
        safe_histories[name] = {
            k: [v.cpu().item() if torch.is_tensor(v) else v for v in vals] 
            for k, vals in history.items()
        }
        
    plt.figure(figsize=(20, 14))
    metrics_to_plot = [
        ('val_total_loss', 'Validation Total Loss', 'Loss'),
        ('val_nsp_loss', 'Validation NSP Loss', 'Loss'),
        ('val_mlm_loss', 'Validation MLM Loss', 'Loss'),
        ('val_nsp_acc', 'Validation NSP Accuracy', 'Accuracy'),
        ('val_mlm_acc', 'Validation MLM Accuracy', 'Accuracy'),
        ('val_mlm_ppl', 'Validation MLM PPL', 'Perplexity'),
        ('val_nsp_f1', 'Validation NSP F1 Score', 'F1 Score'),
        ('val_nsp_roc_auc', 'Validation NSP ROC-AUC', 'ROC-AUC')
    ]
    
    for idx, (metric_key, title, ylabel) in enumerate(metrics_to_plot, 1):
        plt.subplot(3, 3, idx)
        for model_name, history in safe_histories.items():
            if metric_key in history:
                plt.plot(history[metric_key], label=model_name, marker='o' if model_name=='v1' else '^')
        plt.xlabel('Epoch')
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

In [ ]:
all_histories = {
    'v1 (MLM Loss Scale: 20)': history_v1,
    'v2 (MLM Loss Scale: 1)': history_v2,
    'v3 (MLM Loss Scale: 10)': history_v3,
    'v4 (MLM Loss Scale: 50)': history_v4
}
compare_imshow(all_histories)

![](./output/output1.png)

### v1, v2, v3, v4 비교 분석
- NSP (Next Sentence Prediction)관련 지표 (NSP Loss, NSP Accuracy, NSP ROC-AUC):
    - MLM Scale이 1로 가장 낮은 v2 모델(주황색)이 NSP 관련 모든 지표에서 압도적인 1위입니다. NSP Loss가 가장 낮게 떨어지며, NSP Accuracy(~0.68 이상) 및 ROC-AUC(~0.80)가 가장 높습니다.
    - 반면 MLM Scale이 50으로 가장 높은 v4 모델(빨간색)은 NSP Loss가 가장 높고 Accuracy는 가장 낮아, NSP 태스크 학습에 큰 어려움을 겪고 있음이 드러납니다.
- MLM (Masked Language Modeling) 관련 지표 (MLM Loss, MLM Accuracy, MLM PPL):
    - 반대 양상이 나타납니다. v1(Scale 20), v3(Scale 10), v4(Scale 50) 세 모델은 그래프가 서로 가깝게 뭉쳐서 우수한 성능(낮은 Loss 및 PPL, 높은 Accuracy)을 보여줍니다.
    - 하지만 NSP를 가장 잘했던 v2 모델(주황색)은 MLM 지표에서 유독 성능이 떨어지며 그래프가 위쪽(혹은 아래쪽)으로 붕 떠 있는 것을 확인할 수 있습니다.
- Total Loss 및 NSP F1 Score:
    - Total Loss는 MLM Loss의 절대적인 수치에 더 크게 영향을 받아, MLM 최적화를 상대적으로 덜 한 v2가 가장 높게 기록된 것으로 보입니다.
    - NSP F1 Score는 에포크마다 위아래로 진동(Fluctuation)이 심해, 이 그래프만으로는 특정 모델의 일관된 우위를 뚜렷하게 단정하기 어렵습니다.

결론: MLM 스케일이 1인 v2는 문장 관계(NSP) 학습에 뛰어난 반면, 스케일이 10 이상으로 커진 v3, v1, v4는 빈칸 추론(MLM)에 학습이 편향되어 NSP 성능이 크게 희생되는 뚜렷한 트레이드오프(Trade-off) 현상을 보여줍니다.

In [ ]:
all_histories = {
    'v2 (MLM Loss Scale: 1)': history_v2,
    'v5 (MLM Loss Scale: 1.0 + Dynamic)': history_v5
}
compare_imshow(all_histories)

![](./output/output2.png)

### v2, v5 비교 분석
- Total Loss 및 MLM 관련 지표 (MLM Loss, MLM Accuracy, MLM PPL): 
    - 고정 스케일 1을 사용한 v2 모델은 에포크가 지날수록 꾸준히 오차가 감소하며 학습이 안정적으로 진행되지만, 동적 스케일링을 적용한 v5 모델은 에포크 4 이후로 MLM Loss가 약 6.8 근처에서 정체되고 정확도도 더 이상 오르지 않는 한계를 보입니다.

- NSP 관련 지표 (NSP Loss, NSP Accuracy, NSP ROC-AUC): 
    - v5 모델은 에포크 3~4 구간까지는 v2보다 높은 Accuracy와 ROC-AUC(약 0.80)를 기록하며 학습이 잘 되는 듯 보였으나, 그 직후 NSP Loss가 0.60 이상으로 비정상적으로 급증하며 성능이 완전히 무너지는 진동 현상이 발생합니다.

- NSP F1 Score: 
    - 두 모델 모두 에포크마다 위아래로 폭이 큰 진동을 보여 이 그래프만으로는 특정 모델의 일관된 우위를 뚜렷하게 단정하기 어렵습니다.

결론: 고정 스케일을 사용한 v2가 안정적으로 오차를 줄여나가는 것과 달리, 동적 스케일링을 적용한 v5는 학습 중반 이후 가중치를 찾는 과정에서 오히려 NSP 오차가 비정상적으로 급증하며 모델 학습이 붕괴되는 부작용을 겪었습니다.


In [ ]:
all_histories = {
    'v2 (Batch Size: 64)': history_v2,
    'v6 (Batch Size: 256(64*4))': history_v6
}
compare_imshow(all_histories)

![](./output/output3.png)

### v2, v6 비교 분석
- MLM 관련 지표 (Total Loss, MLM Loss, MLM Accuracy, MLM PPL):
    - 기본 배치 사이즈인 v2 모델이 v6 모델보다 훨씬 더 가파르게 오차를 줄이며, MLM Accuracy에서도 약 0.21 수준까지 도달해 압도적으로 우수한 성능을 보여줍니다. 반면 배치를 키운 v6 모델은 MLM 학습 속도가 더디며 Loss가 일찍 정체되는 한계를 보입니다.
- NSP 관련 지표 (NSP Accuracy, NSP ROC-AUC):
    - MLM 결과와는 반대로, 배치 사이즈가 큰 v6 모델이 v2 모델보다 지속적으로 근소하게 더 높은 Accuracy(약 0.69 근처)와 ROC-AUC를 기록하고 있습니다. 이는 큰 배치 사이즈가 문장 간의 전체적인 관계(NSP)를 파악하는 데는 약간 더 유리하게 작용했음을 보여줍니다.
- NSP F1 Score 및 NSP Loss: 두 모델 모두 NSP Loss가 에포크 4~5 부근에서 최저점을 찍은 뒤 살짝 반등하는 추세이며, NSP F1 Score는 위아래로 심하게 요동치고 있어 이 지표들만으로는 유의미한 우위를 확정하기 어렵습니다.

결론: 누적 기울기를 통해 배치 사이즈를 256으로 늘린 v6 모델은 문장 관계 파악(NSP)에서 미세한 성능 향상을 이뤘지만, 빈칸 추론(MLM)의 수렴 속도와 최종 성능을 크게 저하시키는 결과를 낳았기 때문에, 현재 설정에서는 기본 배치 사이즈인 64를 사용한 v2가 더 빠르고 균형 잡힌 학습을 보여줍니다.

In [ ]:
all_histories = {
    'v2 (data: 355555)': history_v2,
    'v7 (data: 128000)': history_v7
}
compare_imshow(all_histories)

![](./output/output4.png)

### v2, v7 비교 분석
- MLM 관련 지표 (Total Loss, MLM Loss, MLM Accuracy, MLM PPL): 
    - 데이터가 충분한 v2 모델은 에포크가 진행될수록 Loss와 PPL이 꾸준히 감소하고 MLM Accuracy가 0.21 부근까지 지속 상승하지만, 데이터가 적은 v7 모델은 에포크 4 이후 오차 감소가 멈추고 학습이 일찍 정체되는 한계를 보입니다.

- NSP 관련 지표 (NSP Loss, NSP Accuracy, NSP ROC-AUC): 
    - v2 모델은 NSP Accuracy가 약 0.68, ROC-AUC가 약 0.80에 도달하며 안정적인 궤도에 오릅니다. 반면 v7 모델은 에포크 4를 기점으로 NSP Loss가 오히려 치솟고 정확도가 하락하고 있어, 데이터 부족에 따른 과적합(Overfitting) 양상을 겪고 있는 것으로 판단됩니다.

- NSP F1 Score: 
    - 두 모델 모두 에포크마다 위아래로 큰 변동성을 보이지만, 전반적인 수치 대역에서 v2 모델이 v7 모델보다 훨씬 높은 점수를 유지하며 확고한 우위를 점하고 있습니다. 
    
결론: 데이터 크기를 128,000개로 축소한 v7 모델은 학습 중반부터 과적합과 성능 정체가 발생한 반면, 355,555개의 넉넉한 데이터를 학습한 v2 모델이 빈칸 추론(MLM)과 문장 관계 파악(NSP) 모든 지표에서 압도적이고 안정적인 성능을 달성했습니다.

### test

In [17]:
import pandas as pd
from IPython.display import display

In [ ]:
def run_inference(model, vocab, sent_a, sent_b):
    """
    주어진 두 문장에 대해 MLM(빈칸 채우기)과 NSP(다음 문장 예측)를 추론합니다.
    [MASK]가 없을 경우 MLM은 건너뛰고 NSP만 추론합니다.
    """
    model.eval() 
    
    pieces_a = vocab.encode_as_pieces(sent_a)
    pieces_b = vocab.encode_as_pieces(sent_b)
    
    # [MASK] 위치 찾기 (없으면 -1 유지)
    mask_idx = -1
    for i, piece in enumerate(pieces_a):
        if piece == '[MASK]':
            mask_idx = i + 1 
            break
            
    if mask_idx == -1: 
        for i, piece in enumerate(pieces_b):
            if piece == '[MASK]':
                mask_idx = len(pieces_a) + 2 + i 
                break
                
    # 모델 입력 텐서 생성
    tokens = ["[CLS]"] + pieces_a + ["[SEP]"] + pieces_b + ["[SEP]"]
    segment = [0] * (len(pieces_a) + 2) + [1] * (len(pieces_b) + 1)
    
    enc_token = [vocab.piece_to_id(t) for t in tokens]
    
    pad_len = config.n_seq - len(enc_token)
    enc_token += [0] * pad_len
    segment += [0] * pad_len
    
    enc_token_t = torch.tensor([enc_token]).long().to(device)
    segment_t = torch.tensor([segment]).long().to(device)
    
    # 모델 추론
    with torch.no_grad():
        logits_nsp, logits_mlm = model(enc_token_t, segment_t)
        
    # 1. NSP 예측 (항상 실행)
    nsp_pred = logits_nsp.argmax(dim=-1).item()
    is_next = "O (이어짐)" if nsp_pred == 1 else "X (무관함)"
    
    # 2. MLM 예측 ([MASK]가 있을 때만 실행)
    if mask_idx != -1:
        mlm_pred_id = logits_mlm[0, mask_idx].argmax(dim=-1).item()
        mlm_pred_token = vocab.id_to_piece(mlm_pred_id)
        mlm_pred_token = mlm_pred_token.replace(' ', '')
    else:
        mlm_pred_token = "마스크 없음" # MASK가 없으면 건너뜀
        
    return mlm_pred_token, is_next

def evaluate_model_inference(model_name, model, sentence_cases):
    """
    모델 이름(예: 'v1')을 입력받아 추론 및 채점 결과를 출력하는 함수입니다.
    """
    
    # 2. 카운트 변수 초기화
    correct_nsp_cnt = 0
    wrong_nsp_cnt = 0
    results = []
    
    # 3. 추론 진행 및 결과 수집
    for i, (sent_a, sent_b, expectation) in enumerate(sentence_cases):
        pred_mask, pred_nsp = run_inference(model, vocab, sent_a, sent_b)

        pred_mark = "O" if "O" in str(pred_nsp).upper() else "X"
        exp_mark = "O" if "O" in str(expectation).upper() else "X"

        if pred_mark == exp_mark:
            correct_nsp_cnt += 1
            is_correct = "정답"
        else:
            wrong_nsp_cnt += 1
            is_correct = "오답"
            
        results.append({
            "No.": i + 1,
            "문장 A": sent_a,
            "문장 B": sent_b,
            "예측 [MASK]": pred_mask,
            "예측 NSP": pred_nsp,
            "기대치": expectation,
            "NSP 채점": is_correct
        })

    # 4. Pandas DataFrame 시각화
    df = pd.DataFrame(results)
    display(df.style.set_properties({
        'text-align': 'left', 
        'white-space': 'pre-wrap'
    }))

    # 5. 요약 통계
    total_cases = len(sentence_cases)
    print(f"\n{'='*35}")
    print(f"[{model_name}] NSP 추론 결과 요약")
    print(f"{'='*35}")
    print(f"- 총 테스트 문장 : {total_cases}개")
    print(f"- 맞은 개수      : {correct_nsp_cnt}개")
    print(f"- 틀린 개수      : {wrong_nsp_cnt}개")
    if total_cases > 0:
        print(f"- NSP 정확도     : {(correct_nsp_cnt / total_cases) * 100:.1f}%")
    print(f"{'='*35}\n")

In [ ]:
from transformers import AutoTokenizer, AutoModelForPreTraining

def run_hf_inference(hf_model, hf_tokenizer, sent_a, sent_b):
    """
    허깅페이스 BERT 모델 전용 추론 함수 (MLM & NSP)
    """
    hf_model.eval()
    
    # 허깅페이스 토크나이저는 문장 2개를 넣으면 알아서 [CLS] A [SEP] B [SEP] 구조와 Token Type ID를 만들어줍니다.
    inputs = hf_tokenizer(sent_a, sent_b, return_tensors="pt", padding="max_length", max_length=128, truncation=True)
    inputs = {k: v.to(hf_model.device) for k, v in inputs.items()}
    
    # [MASK] 토큰 위치 찾기
    input_ids = inputs["input_ids"][0].tolist()
    mask_token_id = hf_tokenizer.mask_token_id
    
    mask_idx = -1
    if mask_token_id in input_ids:
        mask_idx = input_ids.index(mask_token_id)
        
    with torch.no_grad():
        outputs = hf_model(inputs)
        
    # 1. NSP 예측 (허깅페이스 BERT의 NSP 라벨은 0이 '이어짐(IsNext)', 1이 '안 이어짐(NotNext)' 입니다)
    nsp_pred_id = outputs.seq_relationship_logits.argmax(dim=-1).item()
    is_next = "O (이어짐)" if nsp_pred_id == 0 else "X (무관함)"
    
    # 2. MLM 예측
    if mask_idx != -1:
        mlm_pred_id = outputs.prediction_logits[0, mask_idx].argmax(dim=-1).item()
        # 허깅페이스 토크나이저 특성상 앞에 붙는 ## 등의 기호 제거
        mlm_pred_token = hf_tokenizer.convert_ids_to_tokens(mlm_pred_id).replace('##', '')
    else:
        mlm_pred_token = "마스크 없음"
        
    return mlm_pred_token, is_next

def evaluate_hf_model(model_name_or_path, sentence_cases):
    """
    AutoModelForPreTraining을 사용하여 어떤 구조의 허깅페이스 모델이든 자동으로 로딩합니다.
    """
    print(f"=== [HF Base: {model_name_or_path}] 모델 로딩 및 추론 시작 ===")
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # [수정된 부분] 토크나이저와 모델 모두 Auto 클래스를 사용하여 완벽한 호환성 확보
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    model = AutoModelForPreTraining.from_pretrained(model_name_or_path).to(device)
    
    correct_nsp_cnt = 0
    wrong_nsp_cnt = 0
    results = []
    
    for i, (sent_a, sent_b, expectation) in enumerate(sentence_cases):
        pred_mask, pred_nsp = run_hf_inference(model, tokenizer, sent_a, sent_b)

        # O/X 채점 스캐너 로직
        pred_mark = "O" if "O" in str(pred_nsp).upper() else "X"
        exp_mark = "O" if "O" in str(expectation).upper() else "X"

        if pred_mark == exp_mark:
            correct_nsp_cnt += 1
            is_correct = "정답"
        else:
            wrong_nsp_cnt += 1
            is_correct = "오답"
            
        results.append({
            "No.": i + 1,
            "문장 A": sent_a,
            "문장 B": sent_b,
            "예측 [MASK]": pred_mask,
            "예측 NSP": pred_nsp,
            "기대치": expectation,
            "NSP 채점": is_correct
        })

    # Pandas DataFrame 시각화
    import pandas as pd
    from IPython.display import display
    df = pd.DataFrame(results)
    display(df.style.set_properties({
        'text-align': 'left', 
        'white-space': 'pre-wrap'
    }))

    # 요약 통계
    total_cases = len(sentence_cases)
    print(f"\n{'='*35}")
    print(f"[{model_name_or_path}] NSP 추론 결과 요약")
    print(f"{'='*35}")
    print(f"- 총 테스트 문장 : {total_cases}개")
    print(f"- 맞은 개수      : {correct_nsp_cnt}개")
    print(f"- 틀린 개수      : {wrong_nsp_cnt}개")
    if total_cases > 0:
        print(f"- NSP 정확도     : {(correct_nsp_cnt / total_cases) * 100:.1f}%")
    print(f"{'='*35}\n")

### Step 4: 정성적 평가 (Inference Test) 및 KLUE-BERT 비교 분석

정량적인 Loss와 Accuracy 수치만으로는 모델의 실제 문맥 이해 능력을 온전히 파악하기 어렵습니다. 따라서 다양한 상황을 가정한 5가지 카테고리의 테스트 케이스를 구성하여, 직접 학습시킨 모델(v1~v7)들과 사전학습된 대형 모델인 KLUE-BERT(`klue/bert-base`)의 추론 결과를 비교 분석했습니다.

1. 테스트 케이스 구성 (총 25문장)
* 유형 1 (기본 지식): "한국의 수도는 [MASK] 이다." 등 사실 기반의 지식
* 유형 2 (일상 문장): "비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다." 등 보편적 상황
* 유형 3 (논리/인과관계): 원인과 결과에 따른 단어 추론 및 문장 이어짐 판단
* 유형 4 (감정 파악): 기쁨, 슬픔, 배신감 등 화자의 감정 상태 이해도
* 유형 5 (이상치/Red Teaming): 넌센스, 오타, 맥락이 파괴된 문장에서의 방어 능력

2. KLUE-BERT vs mini BERT 비교 결과 요약
* KLUE-BERT (`klue/bert-base`):
  * 파라미터가 압도적으로 큰 만큼, 넌센스나 논리 파악 등 까다로운 케이스에서도 상대적으로 높은 정답률(21개 정답, 84.0%)을 보이며 문맥을 깊이 있게 이해하는 모습을 보였습니다.
* mini BERT (v1 ~ v7):
  * 1M 사이즈의 한계로 인해 복잡한 논리나 넌센스 케이스 방어에는 다소 취약했습니다.
  * 하지만 "한국의 수도"나 "비 올 때 쓰는 물건" 같은 단순 지식 및 일상 문장에 대한 [MASK] 추론과 문장 연속성(NSP) 판단은 나름대로 수행해 내는 모습을 보였습니다.
  * 특히 정량 지표가 가장 안정적이었던 모델(예: v2)에서 가장 상식에 가까운 추론 결과가 도출되는 것을 확인할 수 있었습니다.

3. 정성적 평가 결론
파라미터가 약 1M에 불과한 작은 모델임에도 기본적인 한국어 문법과 자주 등장하는 단어 간의 관계(Co-occurrence)를 일부 모사하는 데 성공했습니다. 다만, 더 깊은 논리적 추론이나 방대한 도메인 지식을 갖추기 위해서는 파라미터 크기의 확장과 훨씬 더 많은 데이터 학습이 필수적임을 확인했습니다.

In [23]:
test_cases = [
    # [1] 기본 지식 및 일반 문장
    ("한국의 수도는 [MASK] 이다.", "서울은 한강을 끼고 있다.", "MLM: 서울 / NSP: O"),
    ("이순신은 조선 시대의 [MASK] 이다.", "거북선을 만들어 왜군을 물리쳤다.", "MLM: 장군 / NSP: O"),
    ("딥러닝 모델은 수많은 데이터를 학습한다.", "이를 통해 새로운 패턴을 추론할 수 있다.", "마스크 없음 / NSP: O"),
    ("물은 수소와 산소로 이루어져 있다.", "오늘 저녁은 치킨을 먹을 것이다.", "마스크 없음 / NSP: X"),
    ("오늘 아침에 늦잠을 자서 지각할 뻔했다.", "다행히 택시를 타서 제시간에 도착했다.", "마스크 없음 / NSP: O"),

    # [2] 간단한 일상 문장
    ("나는 아침에 일어나서 시원한 [MASK] 를 마신다.", "커피는 잠을 깨우는 데 좋다.", "MLM: 물(또는 커피) / NSP: O"),
    ("비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다.", "내일은 날씨가 맑았으면 좋겠다.", "MLM: 우산 / NSP: O"),
    ("배가 너무 고파서 식당에 들어가 김치찌개를 시켰다.", "어제 새로 산 신발이 너무 마음에 든다.", "마스크 없음 / NSP: X"),
    ("주말에 친구들과 제주도로 여행을 가기로 했다.", "비행기 표와 숙소를 미리 예약해 두었다.", "마스크 없음 / NSP: O"),
    ("날씨가 너무 추워서 두꺼운 패딩을 입었다.", "냉장고에서 차가운 아이스크림을 꺼내 먹었다.", "마스크 없음 / NSP: X"),

    # [3] 생각이 필요한 문장 (논리, 인과관계)
    ("수요가 폭발적으로 증가하는데 공급이 부족해지면 물건의 가격은 [MASK] 한다.", "이것은 경제학의 기본 원리이다.", "MLM: 상승 / NSP: O"),
    ("밤하늘에 별이 반짝이는 이유는 별 내부에서 핵융합 반응이 [MASK] 때문이야.", "우주는 정말 신비로운 공간이다.", "MLM: 일어나기 / NSP: O"),
    ("그가 범인이 아니라는 완벽한 알리바이가 [MASK] .", "따라서 경찰은 진범을 다시 수사해야 한다.", "MLM: 증명(또는 있다) / NSP: O"),
    ("만약 지구의 모든 빙하가 녹는다면 해수면이 상승할 것이다.", "이는 해안가 도시들에 큰 위협이 된다.", "마스크 없음 / NSP: O"),
    ("수요가 증가하는데 공급이 부족해지면 어떻게 될까?", "일반적으로 물건의 가격이 상승하게 된다.", "마스크 없음 / NSP: O"),

    # [4] 감정 관련 문장 (Red Teaming)
    ("하루 종일 아무도 내 연락을 받지 않아서 마음이 너무 [MASK] .", "그래서 혼자 방에서 울었다.", "MLM: 슬프다 / NSP: O"),
    ("어려운 시험에서 만점을 받아서 기분이 정말 [MASK] !", "친구들이 다 함께 축하해 주었다.", "MLM: 좋다 / NSP: O"),
    ("네가 나한테 어떻게 그럴 수 있어? 정말 실망스러워.", "다시는 너와 말하고 싶지 않다.", "마스크 없음 / NSP: O"),
    ("어두운 골목길을 혼자 걸어가려니 등골이 오싹하고 너무 무섭다.", "갑자기 검은 고양이가 튀어나왔다.", "마스크 없음 / NSP: O"),
    ("믿었던 친구에게 배신을 당해서 마음이 아프다.", "오늘은 기분이 좋아서 노래방에 갔다.", "마스크 없음 / NSP: X"),

    # [5] 이상치 (넌센스, 오타, 맥락 파괴)
    ("냉장고 문을 열었더니 커다란 [MASK] 가 기타를 치고 있었다.", "코끼리는 코가 길고 귀가 크다.", "MLM: 사람(괴물) / NSP: X"),
    ("나는 오늘 밥을 아주 맛있게 먹었다.", "그래서 갑자기 [MASK] 은 우주로 날아갔다.", "MLM: 로켓 / NSP: X"),
    ("얾뫄가 오뉼 맛있는 [MASK] 를 만드러 주셔따.", "맛이 정말 훌륭해서 두 그릇이나 먹었다.", "MLM: 밥(요리) / NSP: O"),
    ("나는 파란색 사과를 타고 하늘을 날아갔다.", "구름 위에서 피자를 구워 먹었다.", "마스크 없음 / NSP: O (동화적 문맥)"),
    ("세종대왕은 조선의 훌륭한 왕이다.", "사과는 바나나보다 길고 노란색이다.", "마스크 없음 / NSP: X (전혀 무관)"),
]

In [28]:
evaluate_hf_model('klue/bert-base', test_cases)  #skt/kobert-base-v1: 18개(72.0%) / klue/bert-base21개(84.0%) / beomi/kcbert-base 9개(36.0%)

=== [HF Base: klue/bert-base] 모델 로딩 및 추론 시작 ===


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

BertForPreTraining LOAD REPORT from: klue/bert-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,한국의 수도는 [MASK] 이다.,서울은 한강을 끼고 있다.,서울,O (이어짐),MLM: 서울 / NSP: O,정답
1,2,이순신은 조선 시대의 [MASK] 이다.,거북선을 만들어 왜군을 물리쳤다.,명장,O (이어짐),MLM: 장군 / NSP: O,정답
2,3,딥러닝 모델은 수많은 데이터를 학습한다.,이를 통해 새로운 패턴을 추론할 수 있다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
3,4,물은 수소와 산소로 이루어져 있다.,오늘 저녁은 치킨을 먹을 것이다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답
4,5,오늘 아침에 늦잠을 자서 지각할 뻔했다.,다행히 택시를 타서 제시간에 도착했다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
5,6,나는 아침에 일어나서 시원한 [MASK] 를 마신다.,커피는 잠을 깨우는 데 좋다.,커피,O (이어짐),MLM: 물(또는 커피) / NSP: O,정답
6,7,비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다.,내일은 날씨가 맑았으면 좋겠다.,는,O (이어짐),MLM: 우산 / NSP: O,정답
7,8,배가 너무 고파서 식당에 들어가 김치찌개를 시켰다.,어제 새로 산 신발이 너무 마음에 든다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답
8,9,주말에 친구들과 제주도로 여행을 가기로 했다.,비행기 표와 숙소를 미리 예약해 두었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
9,10,날씨가 너무 추워서 두꺼운 패딩을 입었다.,냉장고에서 차가운 아이스크림을 꺼내 먹었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답



[klue/bert-base] NSP 추론 결과 요약
- 총 테스트 문장 : 25개
- 맞은 개수      : 21개
- 틀린 개수      : 4개
- NSP 정확도     : 84.0%



In [29]:
model_name = 'v1'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, test_cases)

=== [v1] 모델 추론 테스트 시작 ===
'./models/bert_v1.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,한국의 수도는 [MASK] 이다.,서울은 한강을 끼고 있다.,[UNK],O (이어짐),MLM: 서울 / NSP: O,정답
1,2,이순신은 조선 시대의 [MASK] 이다.,거북선을 만들어 왜군을 물리쳤다.,[UNK],X (무관함),MLM: 장군 / NSP: O,오답
2,3,딥러닝 모델은 수많은 데이터를 학습한다.,이를 통해 새로운 패턴을 추론할 수 있다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
3,4,물은 수소와 산소로 이루어져 있다.,오늘 저녁은 치킨을 먹을 것이다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답
4,5,오늘 아침에 늦잠을 자서 지각할 뻔했다.,다행히 택시를 타서 제시간에 도착했다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
5,6,나는 아침에 일어나서 시원한 [MASK] 를 마신다.,커피는 잠을 깨우는 데 좋다.,[UNK],X (무관함),MLM: 물(또는 커피) / NSP: O,오답
6,7,비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다.,내일은 날씨가 맑았으면 좋겠다.,[UNK],X (무관함),MLM: 우산 / NSP: O,오답
7,8,배가 너무 고파서 식당에 들어가 김치찌개를 시켰다.,어제 새로 산 신발이 너무 마음에 든다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답
8,9,주말에 친구들과 제주도로 여행을 가기로 했다.,비행기 표와 숙소를 미리 예약해 두었다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
9,10,날씨가 너무 추워서 두꺼운 패딩을 입었다.,냉장고에서 차가운 아이스크림을 꺼내 먹었다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답



[v1] NSP 추론 결과 요약
- 총 테스트 문장 : 25개
- 맞은 개수      : 9개
- 틀린 개수      : 16개
- NSP 정확도     : 36.0%



In [30]:
model_name = 'v2'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, test_cases)

=== [v2] 모델 추론 테스트 시작 ===
'./models/bert_v2.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,한국의 수도는 [MASK] 이다.,서울은 한강을 끼고 있다.,[UNK],O (이어짐),MLM: 서울 / NSP: O,정답
1,2,이순신은 조선 시대의 [MASK] 이다.,거북선을 만들어 왜군을 물리쳤다.,[UNK],O (이어짐),MLM: 장군 / NSP: O,정답
2,3,딥러닝 모델은 수많은 데이터를 학습한다.,이를 통해 새로운 패턴을 추론할 수 있다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
3,4,물은 수소와 산소로 이루어져 있다.,오늘 저녁은 치킨을 먹을 것이다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답
4,5,오늘 아침에 늦잠을 자서 지각할 뻔했다.,다행히 택시를 타서 제시간에 도착했다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
5,6,나는 아침에 일어나서 시원한 [MASK] 를 마신다.,커피는 잠을 깨우는 데 좋다.,[UNK],O (이어짐),MLM: 물(또는 커피) / NSP: O,정답
6,7,비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다.,내일은 날씨가 맑았으면 좋겠다.,[UNK],X (무관함),MLM: 우산 / NSP: O,오답
7,8,배가 너무 고파서 식당에 들어가 김치찌개를 시켰다.,어제 새로 산 신발이 너무 마음에 든다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답
8,9,주말에 친구들과 제주도로 여행을 가기로 했다.,비행기 표와 숙소를 미리 예약해 두었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
9,10,날씨가 너무 추워서 두꺼운 패딩을 입었다.,냉장고에서 차가운 아이스크림을 꺼내 먹었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답



[v2] NSP 추론 결과 요약
- 총 테스트 문장 : 25개
- 맞은 개수      : 16개
- 틀린 개수      : 9개
- NSP 정확도     : 64.0%



In [ ]:
model_name = 'v3'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, test_cases)

=== [v3] 모델 추론 테스트 시작 ===
'./models/bert_v3.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,한국의 수도는 [MASK] 이다.,서울은 한강을 끼고 있다.,▁다음,O (이어짐),MLM: 서울 / NSP: O,정답
1,2,이순신은 조선 시대의 [MASK] 이다.,거북선을 만들어 왜군을 물리쳤다.,[UNK],X (무관함),MLM: 장군 / NSP: O,오답
2,3,딥러닝 모델은 수많은 데이터를 학습한다.,이를 통해 새로운 패턴을 추론할 수 있다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
3,4,물은 수소와 산소로 이루어져 있다.,오늘 저녁은 치킨을 먹을 것이다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답
4,5,오늘 아침에 늦잠을 자서 지각할 뻔했다.,다행히 택시를 타서 제시간에 도착했다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
5,6,나는 아침에 일어나서 시원한 [MASK] 를 마신다.,커피는 잠을 깨우는 데 좋다.,[UNK],X (무관함),MLM: 물(또는 커피) / NSP: O,오답
6,7,비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다.,내일은 날씨가 맑았으면 좋겠다.,[UNK],X (무관함),MLM: 우산 / NSP: O,오답
7,8,배가 너무 고파서 식당에 들어가 김치찌개를 시켰다.,어제 새로 산 신발이 너무 마음에 든다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답
8,9,주말에 친구들과 제주도로 여행을 가기로 했다.,비행기 표와 숙소를 미리 예약해 두었다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
9,10,날씨가 너무 추워서 두꺼운 패딩을 입었다.,냉장고에서 차가운 아이스크림을 꺼내 먹었다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답



[v3] NSP 추론 결과 요약
- 총 테스트 문장 : 25개
- 맞은 개수      : 9개
- 틀린 개수      : 16개
- NSP 정확도     : 36.0%



In [ ]:
model_name = 'v4'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, test_cases)

=== [v4] 모델 추론 테스트 시작 ===
'./models/bert_v4.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,한국의 수도는 [MASK] 이다.,서울은 한강을 끼고 있다.,▁뜻은,X (무관함),MLM: 서울 / NSP: O,오답
1,2,이순신은 조선 시대의 [MASK] 이다.,거북선을 만들어 왜군을 물리쳤다.,▁뜻으로,X (무관함),MLM: 장군 / NSP: O,오답
2,3,딥러닝 모델은 수많은 데이터를 학습한다.,이를 통해 새로운 패턴을 추론할 수 있다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
3,4,물은 수소와 산소로 이루어져 있다.,오늘 저녁은 치킨을 먹을 것이다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답
4,5,오늘 아침에 늦잠을 자서 지각할 뻔했다.,다행히 택시를 타서 제시간에 도착했다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
5,6,나는 아침에 일어나서 시원한 [MASK] 를 마신다.,커피는 잠을 깨우는 데 좋다.,[UNK],X (무관함),MLM: 물(또는 커피) / NSP: O,오답
6,7,비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다.,내일은 날씨가 맑았으면 좋겠다.,[UNK],X (무관함),MLM: 우산 / NSP: O,오답
7,8,배가 너무 고파서 식당에 들어가 김치찌개를 시켰다.,어제 새로 산 신발이 너무 마음에 든다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답
8,9,주말에 친구들과 제주도로 여행을 가기로 했다.,비행기 표와 숙소를 미리 예약해 두었다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
9,10,날씨가 너무 추워서 두꺼운 패딩을 입었다.,냉장고에서 차가운 아이스크림을 꺼내 먹었다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답



[v4] NSP 추론 결과 요약
- 총 테스트 문장 : 25개
- 맞은 개수      : 9개
- 틀린 개수      : 16개
- NSP 정확도     : 36.0%



In [ ]:
model_name = 'v5'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, test_cases)

=== [v5] 모델 추론 테스트 시작 ===
'./models/bert_v5.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,한국의 수도는 [MASK] 이다.,서울은 한강을 끼고 있다.,[UNK],O (이어짐),MLM: 서울 / NSP: O,정답
1,2,이순신은 조선 시대의 [MASK] 이다.,거북선을 만들어 왜군을 물리쳤다.,[UNK],O (이어짐),MLM: 장군 / NSP: O,정답
2,3,딥러닝 모델은 수많은 데이터를 학습한다.,이를 통해 새로운 패턴을 추론할 수 있다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
3,4,물은 수소와 산소로 이루어져 있다.,오늘 저녁은 치킨을 먹을 것이다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답
4,5,오늘 아침에 늦잠을 자서 지각할 뻔했다.,다행히 택시를 타서 제시간에 도착했다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
5,6,나는 아침에 일어나서 시원한 [MASK] 를 마신다.,커피는 잠을 깨우는 데 좋다.,[UNK],O (이어짐),MLM: 물(또는 커피) / NSP: O,정답
6,7,비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다.,내일은 날씨가 맑았으면 좋겠다.,[UNK],O (이어짐),MLM: 우산 / NSP: O,정답
7,8,배가 너무 고파서 식당에 들어가 김치찌개를 시켰다.,어제 새로 산 신발이 너무 마음에 든다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답
8,9,주말에 친구들과 제주도로 여행을 가기로 했다.,비행기 표와 숙소를 미리 예약해 두었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
9,10,날씨가 너무 추워서 두꺼운 패딩을 입었다.,냉장고에서 차가운 아이스크림을 꺼내 먹었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답



[v5] NSP 추론 결과 요약
- 총 테스트 문장 : 25개
- 맞은 개수      : 15개
- 틀린 개수      : 10개
- NSP 정확도     : 60.0%



In [ ]:
model_name = 'v6'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, test_cases)

=== [v6] 모델 추론 테스트 시작 ===
'./models/bert_v6.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,한국의 수도는 [MASK] 이다.,서울은 한강을 끼고 있다.,[UNK],O (이어짐),MLM: 서울 / NSP: O,정답
1,2,이순신은 조선 시대의 [MASK] 이다.,거북선을 만들어 왜군을 물리쳤다.,[UNK],O (이어짐),MLM: 장군 / NSP: O,정답
2,3,딥러닝 모델은 수많은 데이터를 학습한다.,이를 통해 새로운 패턴을 추론할 수 있다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
3,4,물은 수소와 산소로 이루어져 있다.,오늘 저녁은 치킨을 먹을 것이다.,마스크 없음,X (무관함),마스크 없음 / NSP: X,정답
4,5,오늘 아침에 늦잠을 자서 지각할 뻔했다.,다행히 택시를 타서 제시간에 도착했다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
5,6,나는 아침에 일어나서 시원한 [MASK] 를 마신다.,커피는 잠을 깨우는 데 좋다.,[UNK],X (무관함),MLM: 물(또는 커피) / NSP: O,오답
6,7,비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다.,내일은 날씨가 맑았으면 좋겠다.,[UNK],O (이어짐),MLM: 우산 / NSP: O,정답
7,8,배가 너무 고파서 식당에 들어가 김치찌개를 시켰다.,어제 새로 산 신발이 너무 마음에 든다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답
8,9,주말에 친구들과 제주도로 여행을 가기로 했다.,비행기 표와 숙소를 미리 예약해 두었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
9,10,날씨가 너무 추워서 두꺼운 패딩을 입었다.,냉장고에서 차가운 아이스크림을 꺼내 먹었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답



[v6] NSP 추론 결과 요약
- 총 테스트 문장 : 25개
- 맞은 개수      : 11개
- 틀린 개수      : 14개
- NSP 정확도     : 44.0%



In [ ]:
model_name = 'v7'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, test_cases)

=== [v7] 모델 추론 테스트 시작 ===
'./models/bert_v7.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,한국의 수도는 [MASK] 이다.,서울은 한강을 끼고 있다.,▁다른,O (이어짐),MLM: 서울 / NSP: O,정답
1,2,이순신은 조선 시대의 [MASK] 이다.,거북선을 만들어 왜군을 물리쳤다.,▁다음과,X (무관함),MLM: 장군 / NSP: O,오답
2,3,딥러닝 모델은 수많은 데이터를 학습한다.,이를 통해 새로운 패턴을 추론할 수 있다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
3,4,물은 수소와 산소로 이루어져 있다.,오늘 저녁은 치킨을 먹을 것이다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답
4,5,오늘 아침에 늦잠을 자서 지각할 뻔했다.,다행히 택시를 타서 제시간에 도착했다.,마스크 없음,X (무관함),마스크 없음 / NSP: O,오답
5,6,나는 아침에 일어나서 시원한 [MASK] 를 마신다.,커피는 잠을 깨우는 데 좋다.,[UNK],X (무관함),MLM: 물(또는 커피) / NSP: O,오답
6,7,비가 많이 와서 밖으로 나갈 때 [MASK] 을 썼다.,내일은 날씨가 맑았으면 좋겠다.,[UNK],X (무관함),MLM: 우산 / NSP: O,오답
7,8,배가 너무 고파서 식당에 들어가 김치찌개를 시켰다.,어제 새로 산 신발이 너무 마음에 든다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답
8,9,주말에 친구들과 제주도로 여행을 가기로 했다.,비행기 표와 숙소를 미리 예약해 두었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: O,정답
9,10,날씨가 너무 추워서 두꺼운 패딩을 입었다.,냉장고에서 차가운 아이스크림을 꺼내 먹었다.,마스크 없음,O (이어짐),마스크 없음 / NSP: X,오답



[v7] NSP 추론 결과 요약
- 총 테스트 문장 : 25개
- 맞은 개수      : 8개
- 틀린 개수      : 17개
- NSP 정확도     : 32.0%



정성적 평가 분석 결과
1. NSP(문장 연속성) 평가 결과
    - 테스트를 진행한 모델 중 v2 모델이 문장의 문맥 연결성을 가장 잘 파악한 것으로 나타났습니다.

    - 총 25개의 테스트 문장 중 16개를 정확히 판별하며 64.0%* 의 정확도를 달성하여, 정량적 지표에서 보였던 NSP 강점이 실제 추론에서도 증명되었습니다.

2. MLM(빈칸 채우기) 평가 결과
    - 전반적으로 1M 사이즈의 소형 모델 특성상 대부분의 모델이 정답을 맞히지 못하고 [UNK] 토큰을 출력하는 등 어휘 생성에 어려움을 겪었습니다.

    - 다만, v3 모델의 경우 경제 관련 문장에서 '[MASK]' 위치에 '역할을'이라는 단어를 생성하고, 핵융합 관련 문장에서는 '있기'라는 표현을 출력하는 등 다른 모델에 비해 상대적으로 구체적인 어휘를 도출하는 모습을 보였습니다.

3. 결론:  
    - 정성적 평가 결과, 문장 연결성을 가장 정확히 파악한 모델은 64.0%의 성공률을 보인 v2였으나, 빈칸 채우기(MLM)에서는 비록 [UNK] 출력이 잦았음에도 특정 문맥에서 유의미한 단어를 생성해낸 v3 모델이 상대적으로 가장 우수한 성능을 보여주었습니다.

---
### Step 5: 학습 데이터 재검증 (Training Data Sanity Check)

모델이 보여준 추론 결과가 단순히 우연에 의한 것인지, 혹은 데이터의 패턴을 실제로 학습한 것인지 확인하기 위해 학습에 사용되었던 데이터 중 10개를 무작위로 추출하여 재검증을 수행했습니다.

1. 검증 목적
* 암기 및 수렴 확인: 모델이 최소한 노출되었던 데이터의 문맥이라도 정확히 파악하고 있는지 확인합니다.
* 우연성 배제: 테스트 데이터에서의 성과가 단순한 운이 아닌, 학습을 통한 가중치 업데이트의 결과임을 증명합니다.

2. 주요 테스트 케이스 (Train Data 내 추출)
* 전문 도메인 지식: 유기 화합물(화학), 제2차 세계 대전(역사), 맥스웰 방정식(물리) 등 학습 말뭉치에 포함되었던 고난도 문장 위주로 구성했습니다.
* 결과 요약: * 학습된 문장임에도 불구하고 복잡한 수식이나 전문 용어가 포함된 [MASK] 채우기에서는 여전히 어려움을 겪는 모습이 관찰되었습니다. (정확하지 않을 수 있습니다)
    * 그러나 문장의 이어짐(NSP)에 있어서는 처음 보는 문장(Test Data)보다 학습 데이터(Train Data)에서 더 높은 일관성을 보여주어, 모델이 데이터의 패턴을 일정 부분 유의미하게 학습했음을 확인했습니다.

3. 정성적 평가 결론 (학습 완성도)
학습 데이터에 대한 재검증 결과, 우리 모델은 데이터를 완벽하게 암기하는 수준에는 도달하지 못했으나, 반복 노출된 문맥에 대해서는 상대적으로 더 낮은 손실(Loss)과 높은 확신도를 보였습니다. 이는 모델이 무작위 결과가 아닌, 데이터에 근거한 가중치 최적화 과정을 정상적으로 거쳤음을 시사합니다.

In [31]:
train_case =[
    ("대부분은 유기 화합물이다. 유기 화합물을 이루는 주된 [MASK] 원소인 탄소는 다른 화학 원소와는 다르게 매우 긴 사슬 형태로 정렬될 수 있다.", "공유 결합은 오비탈이 겹쳐진 결과 두 원자가 전자쌍을 공유하게 되어 생성되는 결합을 의미한다.", "MLM: 화학 / NSP: O"),
    ("화학의 분과는 전통적으로 다음과 같은 5가지로 나눌 수 있으며, 각각의 분과는 더욱 세분화될 수 있다.", "화학은 취급 대상 및 대상의 취급 방법에 따라서 몇 가지 [MASK] 구분될 수 있다.", "MLM: 분과로 / NSP: O"),
    ("서부가 잉구슈 공화국으로 분리되었기 때문에 인구시인들의 수가 절반 가까이 감소하고, 내전과 사회 불안으로 대부분의 러시아인은 체첸 공화국에서 [MASK] 떠났다.", "체첸 공화국, 또는 줄여서 체첸은 러시아의 연방 주체 중 하나인 공화국이다.", "MLM: 대부분 / NSP: O"),
    ("제2차 세계 대전 동안 소비에트 정부는 체첸이 나치군과 협력했다고 맹비난하였다. 스탈린은 체첸 국민 전체에게 카자흐스탄으로의 강제이주를 [MASK].", "그 후 스탈린이 사망한 지 4년 후인 1957년에 이르러서야 체첸인의 귀환이 허용되었다.", "MLM: 명령했다 / NSP: O"),
    ("화학에서의 많은 실험방법 등은 분석화학에 있어서 중요한 측정기기 등을 개발하는 [MASK].", "오늘 저녁에는 친구들과 함께 맛있는 피자를 먹으러 갈 계획이다.", "MLM: 것이다 / NSP: X"),
    ("맥스웰 방정식은 전기장과 자기장, 전하 밀도와 전류 밀도의 형성을 나타내는 4개의 편미분 [MASK].", "최근 인공지능 기술의 발달로 자율주행 자동차의 상용화가 한층 가까워졌다.", "MLM: 방정식이다 / NSP: X"),
    ("맥스웰의 방정식은 네 개의 법칙을 모아 종합하여 구성한 것으로, 빛과 같은 전자기파의 특성을 [MASK].", "매일 아침 가벼운 조깅을 하는 것은 심혈관 건강을 유지하는 데 큰 도움이 된다.", "MLM: 설명한다 / NSP: X"),
    ("자극의 세기 단위는 웨버(Wb)로, 쿨롱은 세기가 같은 두 개의 자극을 1m 떨어뜨려 놓았을 때 작용하는 힘의 크기가 같은 경우를 1Wb로 [MASK].", "밤하늘에 빛나는 수많은 별들은 각각 다른 크기와 온도를 가진 항성들이다.", "MLM: 정의했다 / NSP: X"),
]

In [32]:
evaluate_hf_model('klue/bert-base', train_case)

=== [HF Base: klue/bert-base] 모델 로딩 및 추론 시작 ===


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

BertForPreTraining LOAD REPORT from: klue/bert-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,대부분은 유기 화합물이다. 유기 화합물을 이루는 주된 [MASK] 원소인 탄소는 다른 화학 원소와는 다르게 매우 긴 사슬 형태로 정렬될 수 있다.,공유 결합은 오비탈이 겹쳐진 결과 두 원자가 전자쌍을 공유하게 되어 생성되는 결합을 의미한다.,구성,O (이어짐),MLM: 화학 / NSP: O,정답
1,2,"화학의 분과는 전통적으로 다음과 같은 5가지로 나눌 수 있으며, 각각의 분과는 더욱 세분화될 수 있다.",화학은 취급 대상 및 대상의 취급 방법에 따라서 몇 가지 [MASK] 구분될 수 있다.,로,O (이어짐),MLM: 분과로 / NSP: O,정답
2,3,"서부가 잉구슈 공화국으로 분리되었기 때문에 인구시인들의 수가 절반 가까이 감소하고, 내전과 사회 불안으로 대부분의 러시아인은 체첸 공화국에서 [MASK] 떠났다.","체첸 공화국, 또는 줄여서 체첸은 러시아의 연방 주체 중 하나인 공화국이다.",멀리,O (이어짐),MLM: 대부분 / NSP: O,정답
3,4,제2차 세계 대전 동안 소비에트 정부는 체첸이 나치군과 협력했다고 맹비난하였다. 스탈린은 체첸 국민 전체에게 카자흐스탄으로의 강제이주를 [MASK].,그 후 스탈린이 사망한 지 4년 후인 1957년에 이르러서야 체첸인의 귀환이 허용되었다.,권한다,O (이어짐),MLM: 명령했다 / NSP: O,정답
4,5,화학에서의 많은 실험방법 등은 분석화학에 있어서 중요한 측정기기 등을 개발하는 [MASK].,오늘 저녁에는 친구들과 함께 맛있는 피자를 먹으러 갈 계획이다.,다,X (무관함),MLM: 것이다 / NSP: X,정답
5,6,"맥스웰 방정식은 전기장과 자기장, 전하 밀도와 전류 밀도의 형성을 나타내는 4개의 편미분 [MASK].",최근 인공지능 기술의 발달로 자율주행 자동차의 상용화가 한층 가까워졌다.,이다,X (무관함),MLM: 방정식이다 / NSP: X,정답
6,7,"맥스웰의 방정식은 네 개의 법칙을 모아 종합하여 구성한 것으로, 빛과 같은 전자기파의 특성을 [MASK].",매일 아침 가벼운 조깅을 하는 것은 심혈관 건강을 유지하는 데 큰 도움이 된다.,나타낸다,X (무관함),MLM: 설명한다 / NSP: X,정답
7,8,"자극의 세기 단위는 웨버(Wb)로, 쿨롱은 세기가 같은 두 개의 자극을 1m 떨어뜨려 놓았을 때 작용하는 힘의 크기가 같은 경우를 1Wb로 [MASK].",밤하늘에 빛나는 수많은 별들은 각각 다른 크기와 온도를 가진 항성들이다.,본다,X (무관함),MLM: 정의했다 / NSP: X,정답



[klue/bert-base] NSP 추론 결과 요약
- 총 테스트 문장 : 8개
- 맞은 개수      : 8개
- 틀린 개수      : 0개
- NSP 정확도     : 100.0%



In [33]:
model_name = 'v1'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, train_case)

=== [v1] 모델 추론 테스트 시작 ===
'./models/bert_v1.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,대부분은 유기 화합물이다. 유기 화합물을 이루는 주된 [MASK] 원소인 탄소는 다른 화학 원소와는 다르게 매우 긴 사슬 형태로 정렬될 수 있다.,공유 결합은 오비탈이 겹쳐진 결과 두 원자가 전자쌍을 공유하게 되어 생성되는 결합을 의미한다.,▁사이의,X (무관함),MLM: 화학 / NSP: O,오답
1,2,"화학의 분과는 전통적으로 다음과 같은 5가지로 나눌 수 있으며, 각각의 분과는 더욱 세분화될 수 있다.",화학은 취급 대상 및 대상의 취급 방법에 따라서 몇 가지 [MASK] 구분될 수 있다.,[UNK],X (무관함),MLM: 분과로 / NSP: O,오답
2,3,"서부가 잉구슈 공화국으로 분리되었기 때문에 인구시인들의 수가 절반 가까이 감소하고, 내전과 사회 불안으로 대부분의 러시아인은 체첸 공화국에서 [MASK] 떠났다.","체첸 공화국, 또는 줄여서 체첸은 러시아의 연방 주체 중 하나인 공화국이다.",[UNK],X (무관함),MLM: 대부분 / NSP: O,오답
3,4,제2차 세계 대전 동안 소비에트 정부는 체첸이 나치군과 협력했다고 맹비난하였다. 스탈린은 체첸 국민 전체에게 카자흐스탄으로의 강제이주를 [MASK].,그 후 스탈린이 사망한 지 4년 후인 1957년에 이르러서야 체첸인의 귀환이 허용되었다.,[UNK],O (이어짐),MLM: 명령했다 / NSP: O,정답
4,5,화학에서의 많은 실험방법 등은 분석화학에 있어서 중요한 측정기기 등을 개발하는 [MASK].,오늘 저녁에는 친구들과 함께 맛있는 피자를 먹으러 갈 계획이다.,[UNK],X (무관함),MLM: 것이다 / NSP: X,정답
5,6,"맥스웰 방정식은 전기장과 자기장, 전하 밀도와 전류 밀도의 형성을 나타내는 4개의 편미분 [MASK].",최근 인공지능 기술의 발달로 자율주행 자동차의 상용화가 한층 가까워졌다.,[UNK],O (이어짐),MLM: 방정식이다 / NSP: X,오답
6,7,"맥스웰의 방정식은 네 개의 법칙을 모아 종합하여 구성한 것으로, 빛과 같은 전자기파의 특성을 [MASK].",매일 아침 가벼운 조깅을 하는 것은 심혈관 건강을 유지하는 데 큰 도움이 된다.,[UNK],X (무관함),MLM: 설명한다 / NSP: X,정답
7,8,"자극의 세기 단위는 웨버(Wb)로, 쿨롱은 세기가 같은 두 개의 자극을 1m 떨어뜨려 놓았을 때 작용하는 힘의 크기가 같은 경우를 1Wb로 [MASK].",밤하늘에 빛나는 수많은 별들은 각각 다른 크기와 온도를 가진 항성들이다.,[UNK],X (무관함),MLM: 정의했다 / NSP: X,정답



[v1] NSP 추론 결과 요약
- 총 테스트 문장 : 8개
- 맞은 개수      : 4개
- 틀린 개수      : 4개
- NSP 정확도     : 50.0%



In [34]:
model_name = 'v2'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, train_case)

=== [v2] 모델 추론 테스트 시작 ===
'./models/bert_v2.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,대부분은 유기 화합물이다. 유기 화합물을 이루는 주된 [MASK] 원소인 탄소는 다른 화학 원소와는 다르게 매우 긴 사슬 형태로 정렬될 수 있다.,공유 결합은 오비탈이 겹쳐진 결과 두 원자가 전자쌍을 공유하게 되어 생성되는 결합을 의미한다.,[UNK],X (무관함),MLM: 화학 / NSP: O,오답
1,2,"화학의 분과는 전통적으로 다음과 같은 5가지로 나눌 수 있으며, 각각의 분과는 더욱 세분화될 수 있다.",화학은 취급 대상 및 대상의 취급 방법에 따라서 몇 가지 [MASK] 구분될 수 있다.,[UNK],X (무관함),MLM: 분과로 / NSP: O,오답
2,3,"서부가 잉구슈 공화국으로 분리되었기 때문에 인구시인들의 수가 절반 가까이 감소하고, 내전과 사회 불안으로 대부분의 러시아인은 체첸 공화국에서 [MASK] 떠났다.","체첸 공화국, 또는 줄여서 체첸은 러시아의 연방 주체 중 하나인 공화국이다.",[UNK],X (무관함),MLM: 대부분 / NSP: O,오답
3,4,제2차 세계 대전 동안 소비에트 정부는 체첸이 나치군과 협력했다고 맹비난하였다. 스탈린은 체첸 국민 전체에게 카자흐스탄으로의 강제이주를 [MASK].,그 후 스탈린이 사망한 지 4년 후인 1957년에 이르러서야 체첸인의 귀환이 허용되었다.,[UNK],X (무관함),MLM: 명령했다 / NSP: O,오답
4,5,화학에서의 많은 실험방법 등은 분석화학에 있어서 중요한 측정기기 등을 개발하는 [MASK].,오늘 저녁에는 친구들과 함께 맛있는 피자를 먹으러 갈 계획이다.,[UNK],X (무관함),MLM: 것이다 / NSP: X,정답
5,6,"맥스웰 방정식은 전기장과 자기장, 전하 밀도와 전류 밀도의 형성을 나타내는 4개의 편미분 [MASK].",최근 인공지능 기술의 발달로 자율주행 자동차의 상용화가 한층 가까워졌다.,[UNK],O (이어짐),MLM: 방정식이다 / NSP: X,오답
6,7,"맥스웰의 방정식은 네 개의 법칙을 모아 종합하여 구성한 것으로, 빛과 같은 전자기파의 특성을 [MASK].",매일 아침 가벼운 조깅을 하는 것은 심혈관 건강을 유지하는 데 큰 도움이 된다.,[UNK],O (이어짐),MLM: 설명한다 / NSP: X,오답
7,8,"자극의 세기 단위는 웨버(Wb)로, 쿨롱은 세기가 같은 두 개의 자극을 1m 떨어뜨려 놓았을 때 작용하는 힘의 크기가 같은 경우를 1Wb로 [MASK].",밤하늘에 빛나는 수많은 별들은 각각 다른 크기와 온도를 가진 항성들이다.,▁등이,O (이어짐),MLM: 정의했다 / NSP: X,오답



[v2] NSP 추론 결과 요약
- 총 테스트 문장 : 8개
- 맞은 개수      : 1개
- 틀린 개수      : 7개
- NSP 정확도     : 12.5%



In [35]:
model_name = 'v3'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, train_case)

=== [v3] 모델 추론 테스트 시작 ===
'./models/bert_v3.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,대부분은 유기 화합물이다. 유기 화합물을 이루는 주된 [MASK] 원소인 탄소는 다른 화학 원소와는 다르게 매우 긴 사슬 형태로 정렬될 수 있다.,공유 결합은 오비탈이 겹쳐진 결과 두 원자가 전자쌍을 공유하게 되어 생성되는 결합을 의미한다.,▁또는,X (무관함),MLM: 화학 / NSP: O,오답
1,2,"화학의 분과는 전통적으로 다음과 같은 5가지로 나눌 수 있으며, 각각의 분과는 더욱 세분화될 수 있다.",화학은 취급 대상 및 대상의 취급 방법에 따라서 몇 가지 [MASK] 구분될 수 있다.,[UNK],X (무관함),MLM: 분과로 / NSP: O,오답
2,3,"서부가 잉구슈 공화국으로 분리되었기 때문에 인구시인들의 수가 절반 가까이 감소하고, 내전과 사회 불안으로 대부분의 러시아인은 체첸 공화국에서 [MASK] 떠났다.","체첸 공화국, 또는 줄여서 체첸은 러시아의 연방 주체 중 하나인 공화국이다.",[UNK],X (무관함),MLM: 대부분 / NSP: O,오답
3,4,제2차 세계 대전 동안 소비에트 정부는 체첸이 나치군과 협력했다고 맹비난하였다. 스탈린은 체첸 국민 전체에게 카자흐스탄으로의 강제이주를 [MASK].,그 후 스탈린이 사망한 지 4년 후인 1957년에 이르러서야 체첸인의 귀환이 허용되었다.,▁않았다,O (이어짐),MLM: 명령했다 / NSP: O,정답
4,5,화학에서의 많은 실험방법 등은 분석화학에 있어서 중요한 측정기기 등을 개발하는 [MASK].,오늘 저녁에는 친구들과 함께 맛있는 피자를 먹으러 갈 계획이다.,▁수,X (무관함),MLM: 것이다 / NSP: X,정답
5,6,"맥스웰 방정식은 전기장과 자기장, 전하 밀도와 전류 밀도의 형성을 나타내는 4개의 편미분 [MASK].",최근 인공지능 기술의 발달로 자율주행 자동차의 상용화가 한층 가까워졌다.,[UNK],X (무관함),MLM: 방정식이다 / NSP: X,정답
6,7,"맥스웰의 방정식은 네 개의 법칙을 모아 종합하여 구성한 것으로, 빛과 같은 전자기파의 특성을 [MASK].",매일 아침 가벼운 조깅을 하는 것은 심혈관 건강을 유지하는 데 큰 도움이 된다.,▁않는다,O (이어짐),MLM: 설명한다 / NSP: X,오답
7,8,"자극의 세기 단위는 웨버(Wb)로, 쿨롱은 세기가 같은 두 개의 자극을 1m 떨어뜨려 놓았을 때 작용하는 힘의 크기가 같은 경우를 1Wb로 [MASK].",밤하늘에 빛나는 수많은 별들은 각각 다른 크기와 온도를 가진 항성들이다.,[UNK],O (이어짐),MLM: 정의했다 / NSP: X,오답



[v3] NSP 추론 결과 요약
- 총 테스트 문장 : 8개
- 맞은 개수      : 3개
- 틀린 개수      : 5개
- NSP 정확도     : 37.5%



In [36]:
model_name = 'v4'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, train_case)

=== [v4] 모델 추론 테스트 시작 ===
'./models/bert_v4.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,대부분은 유기 화합물이다. 유기 화합물을 이루는 주된 [MASK] 원소인 탄소는 다른 화학 원소와는 다르게 매우 긴 사슬 형태로 정렬될 수 있다.,공유 결합은 오비탈이 겹쳐진 결과 두 원자가 전자쌍을 공유하게 되어 생성되는 결합을 의미한다.,[UNK],X (무관함),MLM: 화학 / NSP: O,오답
1,2,"화학의 분과는 전통적으로 다음과 같은 5가지로 나눌 수 있으며, 각각의 분과는 더욱 세분화될 수 있다.",화학은 취급 대상 및 대상의 취급 방법에 따라서 몇 가지 [MASK] 구분될 수 있다.,[UNK],X (무관함),MLM: 분과로 / NSP: O,오답
2,3,"서부가 잉구슈 공화국으로 분리되었기 때문에 인구시인들의 수가 절반 가까이 감소하고, 내전과 사회 불안으로 대부분의 러시아인은 체첸 공화국에서 [MASK] 떠났다.","체첸 공화국, 또는 줄여서 체첸은 러시아의 연방 주체 중 하나인 공화국이다.",[UNK],X (무관함),MLM: 대부분 / NSP: O,오답
3,4,제2차 세계 대전 동안 소비에트 정부는 체첸이 나치군과 협력했다고 맹비난하였다. 스탈린은 체첸 국민 전체에게 카자흐스탄으로의 강제이주를 [MASK].,그 후 스탈린이 사망한 지 4년 후인 1957년에 이르러서야 체첸인의 귀환이 허용되었다.,[UNK],X (무관함),MLM: 명령했다 / NSP: O,오답
4,5,화학에서의 많은 실험방법 등은 분석화학에 있어서 중요한 측정기기 등을 개발하는 [MASK].,오늘 저녁에는 친구들과 함께 맛있는 피자를 먹으러 갈 계획이다.,[UNK],X (무관함),MLM: 것이다 / NSP: X,정답
5,6,"맥스웰 방정식은 전기장과 자기장, 전하 밀도와 전류 밀도의 형성을 나타내는 4개의 편미분 [MASK].",최근 인공지능 기술의 발달로 자율주행 자동차의 상용화가 한층 가까워졌다.,[UNK],X (무관함),MLM: 방정식이다 / NSP: X,정답
6,7,"맥스웰의 방정식은 네 개의 법칙을 모아 종합하여 구성한 것으로, 빛과 같은 전자기파의 특성을 [MASK].",매일 아침 가벼운 조깅을 하는 것은 심혈관 건강을 유지하는 데 큰 도움이 된다.,[UNK],O (이어짐),MLM: 설명한다 / NSP: X,오답
7,8,"자극의 세기 단위는 웨버(Wb)로, 쿨롱은 세기가 같은 두 개의 자극을 1m 떨어뜨려 놓았을 때 작용하는 힘의 크기가 같은 경우를 1Wb로 [MASK].",밤하늘에 빛나는 수많은 별들은 각각 다른 크기와 온도를 가진 항성들이다.,▁수,O (이어짐),MLM: 정의했다 / NSP: X,오답



[v4] NSP 추론 결과 요약
- 총 테스트 문장 : 8개
- 맞은 개수      : 2개
- 틀린 개수      : 6개
- NSP 정확도     : 25.0%



In [37]:
model_name = 'v5'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, train_case)

=== [v5] 모델 추론 테스트 시작 ===
'./models/bert_v5.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,대부분은 유기 화합물이다. 유기 화합물을 이루는 주된 [MASK] 원소인 탄소는 다른 화학 원소와는 다르게 매우 긴 사슬 형태로 정렬될 수 있다.,공유 결합은 오비탈이 겹쳐진 결과 두 원자가 전자쌍을 공유하게 되어 생성되는 결합을 의미한다.,[UNK],O (이어짐),MLM: 화학 / NSP: O,정답
1,2,"화학의 분과는 전통적으로 다음과 같은 5가지로 나눌 수 있으며, 각각의 분과는 더욱 세분화될 수 있다.",화학은 취급 대상 및 대상의 취급 방법에 따라서 몇 가지 [MASK] 구분될 수 있다.,[UNK],O (이어짐),MLM: 분과로 / NSP: O,정답
2,3,"서부가 잉구슈 공화국으로 분리되었기 때문에 인구시인들의 수가 절반 가까이 감소하고, 내전과 사회 불안으로 대부분의 러시아인은 체첸 공화국에서 [MASK] 떠났다.","체첸 공화국, 또는 줄여서 체첸은 러시아의 연방 주체 중 하나인 공화국이다.",[UNK],X (무관함),MLM: 대부분 / NSP: O,오답
3,4,제2차 세계 대전 동안 소비에트 정부는 체첸이 나치군과 협력했다고 맹비난하였다. 스탈린은 체첸 국민 전체에게 카자흐스탄으로의 강제이주를 [MASK].,그 후 스탈린이 사망한 지 4년 후인 1957년에 이르러서야 체첸인의 귀환이 허용되었다.,[UNK],O (이어짐),MLM: 명령했다 / NSP: O,정답
4,5,화학에서의 많은 실험방법 등은 분석화학에 있어서 중요한 측정기기 등을 개발하는 [MASK].,오늘 저녁에는 친구들과 함께 맛있는 피자를 먹으러 갈 계획이다.,[UNK],X (무관함),MLM: 것이다 / NSP: X,정답
5,6,"맥스웰 방정식은 전기장과 자기장, 전하 밀도와 전류 밀도의 형성을 나타내는 4개의 편미분 [MASK].",최근 인공지능 기술의 발달로 자율주행 자동차의 상용화가 한층 가까워졌다.,[UNK],O (이어짐),MLM: 방정식이다 / NSP: X,오답
6,7,"맥스웰의 방정식은 네 개의 법칙을 모아 종합하여 구성한 것으로, 빛과 같은 전자기파의 특성을 [MASK].",매일 아침 가벼운 조깅을 하는 것은 심혈관 건강을 유지하는 데 큰 도움이 된다.,[UNK],O (이어짐),MLM: 설명한다 / NSP: X,오답
7,8,"자극의 세기 단위는 웨버(Wb)로, 쿨롱은 세기가 같은 두 개의 자극을 1m 떨어뜨려 놓았을 때 작용하는 힘의 크기가 같은 경우를 1Wb로 [MASK].",밤하늘에 빛나는 수많은 별들은 각각 다른 크기와 온도를 가진 항성들이다.,[UNK],O (이어짐),MLM: 정의했다 / NSP: X,오답



[v5] NSP 추론 결과 요약
- 총 테스트 문장 : 8개
- 맞은 개수      : 4개
- 틀린 개수      : 4개
- NSP 정확도     : 50.0%



In [38]:
model_name = 'v6'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, train_case)

=== [v6] 모델 추론 테스트 시작 ===
'./models/bert_v6.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,대부분은 유기 화합물이다. 유기 화합물을 이루는 주된 [MASK] 원소인 탄소는 다른 화학 원소와는 다르게 매우 긴 사슬 형태로 정렬될 수 있다.,공유 결합은 오비탈이 겹쳐진 결과 두 원자가 전자쌍을 공유하게 되어 생성되는 결합을 의미한다.,[UNK],O (이어짐),MLM: 화학 / NSP: O,정답
1,2,"화학의 분과는 전통적으로 다음과 같은 5가지로 나눌 수 있으며, 각각의 분과는 더욱 세분화될 수 있다.",화학은 취급 대상 및 대상의 취급 방법에 따라서 몇 가지 [MASK] 구분될 수 있다.,[UNK],O (이어짐),MLM: 분과로 / NSP: O,정답
2,3,"서부가 잉구슈 공화국으로 분리되었기 때문에 인구시인들의 수가 절반 가까이 감소하고, 내전과 사회 불안으로 대부분의 러시아인은 체첸 공화국에서 [MASK] 떠났다.","체첸 공화국, 또는 줄여서 체첸은 러시아의 연방 주체 중 하나인 공화국이다.",[UNK],X (무관함),MLM: 대부분 / NSP: O,오답
3,4,제2차 세계 대전 동안 소비에트 정부는 체첸이 나치군과 협력했다고 맹비난하였다. 스탈린은 체첸 국민 전체에게 카자흐스탄으로의 강제이주를 [MASK].,그 후 스탈린이 사망한 지 4년 후인 1957년에 이르러서야 체첸인의 귀환이 허용되었다.,[UNK],O (이어짐),MLM: 명령했다 / NSP: O,정답
4,5,화학에서의 많은 실험방법 등은 분석화학에 있어서 중요한 측정기기 등을 개발하는 [MASK].,오늘 저녁에는 친구들과 함께 맛있는 피자를 먹으러 갈 계획이다.,[UNK],O (이어짐),MLM: 것이다 / NSP: X,오답
5,6,"맥스웰 방정식은 전기장과 자기장, 전하 밀도와 전류 밀도의 형성을 나타내는 4개의 편미분 [MASK].",최근 인공지능 기술의 발달로 자율주행 자동차의 상용화가 한층 가까워졌다.,[UNK],O (이어짐),MLM: 방정식이다 / NSP: X,오답
6,7,"맥스웰의 방정식은 네 개의 법칙을 모아 종합하여 구성한 것으로, 빛과 같은 전자기파의 특성을 [MASK].",매일 아침 가벼운 조깅을 하는 것은 심혈관 건강을 유지하는 데 큰 도움이 된다.,[UNK],O (이어짐),MLM: 설명한다 / NSP: X,오답
7,8,"자극의 세기 단위는 웨버(Wb)로, 쿨롱은 세기가 같은 두 개의 자극을 1m 떨어뜨려 놓았을 때 작용하는 힘의 크기가 같은 경우를 1Wb로 [MASK].",밤하늘에 빛나는 수많은 별들은 각각 다른 크기와 온도를 가진 항성들이다.,[UNK],O (이어짐),MLM: 정의했다 / NSP: X,오답



[v6] NSP 추론 결과 요약
- 총 테스트 문장 : 8개
- 맞은 개수      : 3개
- 틀린 개수      : 5개
- NSP 정확도     : 37.5%



In [39]:
model_name = 'v7'
print(f"=== [{model_name}] 모델 추론 테스트 시작 ===")

# 1. 모델 초기화 및 가중치 로드
model = PreTrainModel(config).to(device)
model_path = os.path.join(MODELS_DIR, f"bert_{model_name}.pt")

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"'{model_path}' 가중치를 성공적으로 불러왔습니다!\n")

evaluate_model_inference(model_name, model, train_case)

=== [v7] 모델 추론 테스트 시작 ===
'./models/bert_v7.pt' 가중치를 성공적으로 불러왔습니다!



,No.,문장 A,문장 B,예측 [MASK],예측 NSP,기대치,NSP 채점
0,1,대부분은 유기 화합물이다. 유기 화합물을 이루는 주된 [MASK] 원소인 탄소는 다른 화학 원소와는 다르게 매우 긴 사슬 형태로 정렬될 수 있다.,공유 결합은 오비탈이 겹쳐진 결과 두 원자가 전자쌍을 공유하게 되어 생성되는 결합을 의미한다.,[UNK],X (무관함),MLM: 화학 / NSP: O,오답
1,2,"화학의 분과는 전통적으로 다음과 같은 5가지로 나눌 수 있으며, 각각의 분과는 더욱 세분화될 수 있다.",화학은 취급 대상 및 대상의 취급 방법에 따라서 몇 가지 [MASK] 구분될 수 있다.,[UNK],X (무관함),MLM: 분과로 / NSP: O,오답
2,3,"서부가 잉구슈 공화국으로 분리되었기 때문에 인구시인들의 수가 절반 가까이 감소하고, 내전과 사회 불안으로 대부분의 러시아인은 체첸 공화국에서 [MASK] 떠났다.","체첸 공화국, 또는 줄여서 체첸은 러시아의 연방 주체 중 하나인 공화국이다.",[UNK],X (무관함),MLM: 대부분 / NSP: O,오답
3,4,제2차 세계 대전 동안 소비에트 정부는 체첸이 나치군과 협력했다고 맹비난하였다. 스탈린은 체첸 국민 전체에게 카자흐스탄으로의 강제이주를 [MASK].,그 후 스탈린이 사망한 지 4년 후인 1957년에 이르러서야 체첸인의 귀환이 허용되었다.,[UNK],X (무관함),MLM: 명령했다 / NSP: O,오답
4,5,화학에서의 많은 실험방법 등은 분석화학에 있어서 중요한 측정기기 등을 개발하는 [MASK].,오늘 저녁에는 친구들과 함께 맛있는 피자를 먹으러 갈 계획이다.,[UNK],O (이어짐),MLM: 것이다 / NSP: X,오답
5,6,"맥스웰 방정식은 전기장과 자기장, 전하 밀도와 전류 밀도의 형성을 나타내는 4개의 편미분 [MASK].",최근 인공지능 기술의 발달로 자율주행 자동차의 상용화가 한층 가까워졌다.,[UNK],X (무관함),MLM: 방정식이다 / NSP: X,정답
6,7,"맥스웰의 방정식은 네 개의 법칙을 모아 종합하여 구성한 것으로, 빛과 같은 전자기파의 특성을 [MASK].",매일 아침 가벼운 조깅을 하는 것은 심혈관 건강을 유지하는 데 큰 도움이 된다.,[UNK],O (이어짐),MLM: 설명한다 / NSP: X,오답
7,8,"자극의 세기 단위는 웨버(Wb)로, 쿨롱은 세기가 같은 두 개의 자극을 1m 떨어뜨려 놓았을 때 작용하는 힘의 크기가 같은 경우를 1Wb로 [MASK].",밤하늘에 빛나는 수많은 별들은 각각 다른 크기와 온도를 가진 항성들이다.,[UNK],O (이어짐),MLM: 정의했다 / NSP: X,오답



[v7] NSP 추론 결과 요약
- 총 테스트 문장 : 8개
- 맞은 개수      : 1개
- 틀린 개수      : 7개
- NSP 정확도     : 12.5%

